# Bengali MMS CTC final confirmed tiny-overfit test

This is the final post-audit test notebook. It uses the attached local
four-dialect dataset and saved checkpoint, records the user's confirmation for
all 32 audio/transcript pairs, validates the checkpoint contract, and runs a
fresh plain MMS-CTC overfit test. No MoE, dialect loss, augmentation, or DDP is
used. Existing datasets, checkpoints, and earlier notebook outputs are not
modified.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

REPO_COMMIT = "4cb0d20"
REPO_DIR = Path("/kaggle/working/bengali-dialect-asr")
RUN_DIR = Path("/kaggle/working/ctc-collapse-diagnostics")
LOG_DIR = RUN_DIR / "logs"
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Recreate only the small source snapshot needed by the diagnostics. This is
# deliberately local: Kaggle workers sometimes cannot resolve github.com even
# when the notebook has internet enabled.
EMBEDDED_SOURCES = {'scripts/ctc_collapse_diagnostics.py': '#!/usr/bin/env python\n"""Audit CTC wiring, labels, lengths, logits, and decoding for one checkpoint.\n\nThis script is read-only with respect to the dataset and checkpoint.  It writes\nJSON/CSV reports below ``--output-dir`` and never starts training.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport numpy as np\nimport os\nimport random\nimport re\nimport sys\nimport unicodedata\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path, PurePosixPath\nfrom typing import Iterable, Iterator\n\nEXPECTED_BLANK_ID = 0\nEXPECTED_DELIMITER_ID = 2\nEXPECTED_VOCAB_SIZE = 73\nTARGET_SAMPLE_RATE = 16_000\n\nDISTRICT_TO_DIALECT = {\n    "Alipurduar": "Kamrupi",\n    "CoochBehar": "Kamrupi",\n    "Darjeeling": "Kamrupi",\n    "Jalpaiguri": "Kamrupi",\n    "Jhargram": "Jharkhandi",\n    "PaschimMedinipur": "Jharkhandi",\n    "Purulia": "Jharkhandi",\n    "Malda": "Varendri",\n    "DakshinDinajpur": "Varendri",\n    "North24Parganas": "Rarhi",\n    "Kolkata": "Rarhi",\n}\n\nZERO_WIDTH_RE = re.compile(r"[\\u200B-\\u200D\\uFEFF]")\nWHITESPACE_RE = re.compile(r"\\s+")\n\n\ndef normalize_text(value: object) -> str:\n    text = unicodedata.normalize("NFC", str(value or ""))\n    text = ZERO_WIDTH_RE.sub("", text)\n    cleaned = []\n    for character in text:\n        if character.isspace():\n            cleaned.append(" ")\n        elif "\\u0980" <= character <= "\\u09FF" and unicodedata.category(character)[0] in {"L", "M", "N"}:\n            cleaned.append(character)\n        else:\n            cleaned.append(" ")\n    return WHITESPACE_RE.sub(" ", "".join(cleaned)).strip()\n\n\ndef split_layout(root: Path) -> str:\n    if all((root / split).is_dir() for split in ("train", "validation", "test")):\n        return "directories"\n    if all((root / f"{split}.zip").is_file() for split in ("train", "validation", "test")):\n        return "split-zips"\n    raise RuntimeError(\n        f"{root} must contain train/validation/test directories or split ZIP files"\n    )\n\n\ndef iter_pairs(root: Path, split: str) -> Iterator[dict]:\n    """Yield matched WAV/TXT records for exactly the 11 approved districts."""\n    mode = split_layout(root)\n    if mode == "directories":\n        split_dir = root / split\n        for district in DISTRICT_TO_DIALECT:\n            district_dir = split_dir / district\n            if not district_dir.is_dir():\n                raise RuntimeError(f"Missing required district directory: {district_dir}")\n            txt_by_key = {\n                path.relative_to(district_dir).with_suffix("").as_posix(): path\n                for path in district_dir.rglob("*.txt")\n            }\n            wav_by_key = {\n                path.relative_to(district_dir).with_suffix("").as_posix(): path\n                for path in district_dir.rglob("*.wav")\n            }\n            if set(txt_by_key) != set(wav_by_key):\n                raise RuntimeError(\n                    f"WAV/TXT mismatch in {district}: "\n                    f"missing_audio={len(set(txt_by_key) - set(wav_by_key))} "\n                    f"missing_text={len(set(wav_by_key) - set(txt_by_key))}"\n                )\n            for key in sorted(txt_by_key):\n                transcript = normalize_text(\n                    txt_by_key[key].read_text(encoding="utf-8-sig", errors="strict")\n                )\n                if transcript:\n                    yield {\n                        "sample_id": f"{district}/{key}",\n                        "audio": wav_by_key[key],\n                        "transcript": transcript,\n                        "district": district,\n                        "dialect": DISTRICT_TO_DIALECT[district],\n                    }\n        return\n\n    with zipfile.ZipFile(root / f"{split}.zip") as archive:\n        grouped: dict[tuple[str, str], dict[str, str]] = {}\n        for info in archive.infolist():\n            if info.is_dir():\n                continue\n            parts = list(PurePosixPath(info.filename).parts)\n            if parts and parts[0].lower() == split.lower():\n                parts = parts[1:]\n            if len(parts) < 2 or parts[0] not in DISTRICT_TO_DIALECT:\n                continue\n            relative = PurePosixPath(*parts[1:])\n            suffix = relative.suffix.lower()\n            if suffix not in {".wav", ".txt"}:\n                continue\n            grouped.setdefault((parts[0], relative.with_suffix("").as_posix()), {})[\n                suffix\n            ] = info.filename\n        for (district, key), pair in sorted(grouped.items()):\n            if set(pair) != {".wav", ".txt"}:\n                raise RuntimeError(f"ZIP WAV/TXT mismatch: {district}/{key}")\n            transcript = normalize_text(\n                archive.read(pair[".txt"]).decode("utf-8-sig", errors="strict")\n            )\n            if transcript:\n                yield {\n                    "sample_id": f"{district}/{key}",\n                    "audio": (root / f"{split}.zip", pair[".wav"]),\n                    "transcript": transcript,\n                    "district": district,\n                    "dialect": DISTRICT_TO_DIALECT[district],\n                }\n\n\ndef choose_rows(root: Path, split: str, limit: int, seed: int = 42) -> list[dict]:\n    rows = list(iter_pairs(root, split))\n    rows.sort(key=lambda row: row["sample_id"])\n    if limit <= 0 or len(rows) <= limit:\n        return rows\n    groups = {district: [] for district in DISTRICT_TO_DIALECT}\n    for row in rows:\n        groups[row["district"]].append(row)\n    rng = random.Random(seed)\n    for values in groups.values():\n        rng.shuffle(values)\n    selected = []\n    while len(selected) < limit:\n        progressed = False\n        for district in DISTRICT_TO_DIALECT:\n            if groups[district] and len(selected) < limit:\n                selected.append(groups[district].pop())\n                progressed = True\n        if not progressed:\n            break\n    selected.sort(key=lambda row: row["sample_id"])\n    return selected\n\n\ndef make_manifest(data_root: Path, output: Path, count: int = 32, seed: int = 42) -> dict:\n    rows = choose_rows(data_root, "test", max(count, len(DISTRICT_TO_DIALECT)), seed)\n    rows = rows[:count]\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(\n            handle,\n            fieldnames=["sample_id", "audio", "transcript", "district", "dialect", "manually_verified"],\n        )\n        writer.writeheader()\n        for row in rows:\n            audio = row["audio"]\n            if isinstance(audio, tuple):\n                audio = f"{audio[0]}::{audio[1]}"\n            writer.writerow({**row, "audio": str(audio), "manually_verified": "NO"})\n    summary = {\n        "manifest": str(output),\n        "count": len(rows),\n        "district_counts": dict(Counter(row["district"] for row in rows)),\n        "requires_manual_audio_transcript_check": True,\n    }\n    output.with_name("tiny_manifest_summary.json").write_text(\n        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    return summary\n\n\ndef read_audio(source, archive_cache: dict | None = None):\n    import io\n    import soundfile as sf\n    import torch\n    import torchaudio\n\n    if isinstance(source, tuple):\n        archive_cache = archive_cache if archive_cache is not None else {}\n        archive = archive_cache.get(str(source[0]))\n        if archive is None:\n            archive = zipfile.ZipFile(source[0])\n            archive_cache[str(source[0])] = archive\n        handle = io.BytesIO(archive.read(source[1]))\n    else:\n        handle = source\n    waveform, sample_rate = sf.read(handle, dtype="float32", always_2d=True)\n    waveform = waveform.mean(axis=1)\n    if not len(waveform) or not np.isfinite(waveform).all():\n        raise ValueError("Audio is empty or contains non-finite samples")\n    if sample_rate != TARGET_SAMPLE_RATE:\n        waveform = torchaudio.functional.resample(\n            torch.from_numpy(waveform), int(sample_rate), TARGET_SAMPLE_RATE\n        ).numpy()\n    return waveform.astype("float32", copy=False)\n\n\ndef _load_processor(checkpoint: Path, model_name: str):\n    from transformers import Wav2Vec2Processor\n\n    candidates = [checkpoint / "processor", checkpoint]\n    parent = checkpoint.parent\n    for _ in range(3):\n        candidates.append(parent / "processor")\n        parent = parent.parent\n    candidate = next(\n        (path for path in candidates if (path / "processor_config.json").is_file()),\n        checkpoint,\n    )\n    processor = Wav2Vec2Processor.from_pretrained(candidate)\n    tokenizer = processor.tokenizer\n    if len(tokenizer) != EXPECTED_VOCAB_SIZE:\n        raise ValueError(f"Expected Bengali CTC vocabulary {EXPECTED_VOCAB_SIZE}, got {len(tokenizer)}")\n    if tokenizer.pad_token_id != EXPECTED_BLANK_ID:\n        raise ValueError(f"Expected blank/pad ID 0, got {tokenizer.pad_token_id}")\n    if tokenizer.unk_token_id != 1:\n        raise ValueError(f"Expected unknown ID 1, got {tokenizer.unk_token_id}")\n    if tokenizer.convert_tokens_to_ids("|") != EXPECTED_DELIMITER_ID:\n        raise ValueError("Expected word delimiter ID 2")\n    if tokenizer.word_delimiter_token != "|":\n        raise ValueError("Expected word delimiter token \'|\'")\n    feature = processor.feature_extractor\n    if int(feature.sampling_rate) != TARGET_SAMPLE_RATE or not bool(feature.do_normalize):\n        raise ValueError("Checkpoint processor is not the normalized 16-kHz MMS processor")\n    return processor\n\n\ndef _load_model(checkpoint: Path, repo_root: Path, processor, model_name: str, device):\n    import torch\n    from omegaconf import OmegaConf\n    from asr_dialect_benchmark.modeling import BengaliDialectASR\n\n    config_path = checkpoint / "config.json"\n    if not config_path.is_file():\n        raise FileNotFoundError(f"Missing checkpoint config: {config_path}")\n    config_data = json.loads(config_path.read_text(encoding="utf-8"))\n    saved_tokens = int(config_data.get("model", {}).get("num_tokens", EXPECTED_VOCAB_SIZE))\n    if saved_tokens != EXPECTED_VOCAB_SIZE:\n        raise ValueError(\n            "Checkpoint model.num_tokens is not the Bengali CTC size: "\n            f"{saved_tokens}"\n        )\n    config = OmegaConf.create(config_data)\n    config.model.num_tokens = EXPECTED_VOCAB_SIZE\n    config.model.gradient_checkpointing = False\n    model = BengaliDialectASR(config)\n    if model.ctc_head.out_features != EXPECTED_VOCAB_SIZE:\n        raise ValueError(f"CTC head has {model.ctc_head.out_features} outputs")\n\n    state_path = None\n    for candidate in ("model.safetensors", "model_state.pt", "pytorch_model.bin"):\n        if (checkpoint / candidate).is_file():\n            state_path = checkpoint / candidate\n            break\n    if state_path is None:\n        raise FileNotFoundError(f"No model state found in {checkpoint}")\n    if state_path.suffix == ".safetensors":\n        from safetensors.torch import load_file\n\n        state = load_file(str(state_path), device="cpu")\n    else:\n        try:\n            state = torch.load(state_path, map_location="cpu", weights_only=True)\n        except TypeError:\n            state = torch.load(state_path, map_location="cpu")\n    if isinstance(state, dict) and "state_dict" in state and isinstance(state["state_dict"], dict):\n        state = state["state_dict"]\n    state = {\n        (key.removeprefix("module.") if key.startswith("module.") else key): value\n        for key, value in state.items()\n    }\n    missing, unexpected = model.load_state_dict(state, strict=False)\n    if any(key.startswith("ctc_head.") for key in missing):\n        raise ValueError(f"Checkpoint is missing Bengali CTC head weights: {missing}")\n    model.to(device).eval()\n    return model, config_data, {\n        "state_path": str(state_path),\n        "missing_keys": list(missing),\n        "unexpected_keys": list(unexpected),\n    }\n\n\ndef _decode(processor, ids: Iterable[int], blank_id: int) -> str:\n    collapsed = []\n    previous = None\n    for raw in ids:\n        token_id = int(raw)\n        if token_id == blank_id:\n            previous = token_id\n            continue\n        if token_id == previous:\n            continue\n        collapsed.append(token_id)\n        previous = token_id\n    return processor.tokenizer.decode(\n        collapsed, group_tokens=False, skip_special_tokens=True\n    ).strip()\n\n\ndef edit_distance(reference: list, hypothesis: list) -> int:\n    previous = list(range(len(hypothesis) + 1))\n    for i, ref in enumerate(reference, start=1):\n        current = [i]\n        for j, hyp in enumerate(hypothesis, start=1):\n            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ref != hyp)))\n        previous = current\n    return previous[-1]\n\n\ndef _bias_report(model) -> dict:\n    bias = model.ctc_head.bias.detach().float().cpu()\n    values = bias.tolist()\n    largest = sorted(enumerate(values), key=lambda item: item[1], reverse=True)[:10]\n    return {\n        "blank_bias_id_0": float(values[EXPECTED_BLANK_ID]),\n        "delimiter_bias_id_2": float(values[EXPECTED_DELIMITER_ID]),\n        "largest_biases": [[int(index), float(value)] for index, value in largest],\n        "initialization": "torch.nn.Linear default initialization unless checkpoint metadata says otherwise",\n    }\n\n\ndef audit_checkpoint(\n    checkpoint: Path,\n    data_root: Path,\n    repo_root: Path,\n    output_dir: Path,\n    sample_count: int,\n    batch_size: int,\n    model_name: str,\n    seed: int,\n) -> dict:\n    import torch\n\n    sys.path.insert(0, str(repo_root / "src"))\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    processor = _load_processor(checkpoint, model_name)\n    model, saved_config, state_report = _load_model(\n        checkpoint, repo_root, processor, model_name, device\n    )\n    rows = choose_rows(data_root, "validation", sample_count, seed)\n    if not rows:\n        raise RuntimeError("No validation records were selected")\n\n    label_counts = Counter()\n    label_total = 0\n    label_examples = []\n    label_decode_mismatches = []\n    frame_counts = Counter()\n    predictions = []\n    invalid_lengths = []\n    blank_probabilities = []\n    archive_cache = {}\n    for start in range(0, len(rows), max(1, batch_size)):\n        batch_rows = rows[start : start + max(1, batch_size)]\n        arrays = [torch.from_numpy(read_audio(row["audio"], archive_cache)) for row in batch_rows]\n        processed = processor.feature_extractor(\n            [array.numpy() for array in arrays],\n            sampling_rate=TARGET_SAMPLE_RATE,\n            return_tensors="pt",\n            padding=True,\n            return_attention_mask=True,\n        )\n        input_values = processed["input_values"].to(device)\n        attention_mask = processed["attention_mask"].to(device)\n        input_lengths = attention_mask.sum(-1).long()\n        target_lists = [processor.tokenizer(row["transcript"]).input_ids for row in batch_rows]\n        for row, target in zip(batch_rows, target_lists):\n            label_counts.update(int(item) for item in target)\n            label_total += len(target)\n            decoded_target = processor.tokenizer.decode(\n                target, group_tokens=False, skip_special_tokens=True\n            ).strip()\n            if decoded_target != row["transcript"] and len(label_decode_mismatches) < 20:\n                label_decode_mismatches.append(\n                    {\n                        "sample_id": row["sample_id"],\n                        "reference": row["transcript"],\n                        "decoded_target": decoded_target,\n                    }\n                )\n            if len(label_examples) < 5:\n                label_examples.append(\n                    {\n                        "sample_id": row["sample_id"],\n                        "reference": row["transcript"],\n                        "target_ids": target,\n                        "decoded_target": decoded_target,\n                    }\n                )\n        with torch.inference_mode():\n            outputs = model(input_values, attention_mask, input_lengths)\n        logits = outputs["logits"].float()\n        assert logits.ndim == 3\n        assert logits.shape[-1] == EXPECTED_VOCAB_SIZE\n        assert int(processor.tokenizer.pad_token_id) == EXPECTED_BLANK_ID\n        output_lengths = outputs["output_lengths"].long().clamp(0, logits.shape[1])\n        ids = logits.argmax(-1)\n        probabilities = logits.softmax(-1)\n        for row_index, row in enumerate(batch_rows):\n            length = int(output_lengths[row_index].item())\n            sequence = ids[row_index, :length].detach().cpu().tolist()\n            frame_counts.update(sequence)\n            target = target_lists[row_index]\n            repeats = sum(left == right for left, right in zip(target, target[1:]))\n            minimum = len(target) + repeats\n            if length < minimum:\n                invalid_lengths.append(\n                    {\n                        "sample_id": row["sample_id"],\n                        "output_length": length,\n                        "target_length": len(target),\n                        "adjacent_repeats": repeats,\n                        "minimum_required": minimum,\n                    }\n                )\n            probability = probabilities[row_index, :length]\n            blank_probabilities.append(float(probability[:, EXPECTED_BLANK_ID].mean().item()))\n            prediction = _decode(processor, sequence, EXPECTED_BLANK_ID)\n            predictions.append(\n                {\n                    "sample_id": row["sample_id"],\n                    "district": row["district"],\n                    "dialect": row["dialect"],\n                    "reference": row["transcript"],\n                    "prediction": prediction,\n                    "empty_prediction": not bool(prediction),\n                    "output_length": length,\n                    "target_length": len(target),\n                    "top_frame_token_ids": Counter(sequence).most_common(15),\n                }\n            )\n\n    if label_counts[EXPECTED_BLANK_ID]:\n        raise ValueError(\n            f"Valid targets contain CTC blank ID 0: count={label_counts[EXPECTED_BLANK_ID]}"\n        )\n    label_top = [[int(index), int(count)] for index, count in label_counts.most_common(20)]\n    frame_total = max(1, sum(frame_counts.values()))\n    reference_chars = sum(len(row["reference"].replace(" ", "")) for row in predictions)\n    prediction_chars = sum(len(row["prediction"].replace(" ", "")) for row in predictions)\n    cer_distance = sum(\n        edit_distance(list(row["reference"].replace(" ", "")), list(row["prediction"].replace(" ", "")))\n        for row in predictions\n    )\n    wer_distance = sum(\n        edit_distance(row["reference"].split(), row["prediction"].split())\n        for row in predictions\n    )\n    reference_words = sum(len(row["reference"].split()) for row in predictions)\n    report = {\n        "status": "ok",\n        "checkpoint": str(checkpoint),\n        "device": str(device),\n        "model_path": state_report,\n        "saved_model_num_tokens": int(saved_config.get("model", {}).get("num_tokens", -1)),\n        "ctc_contract": {\n            "blank_id": EXPECTED_BLANK_ID,\n            "padding_id": int(processor.tokenizer.pad_token_id),\n            "unknown_id": int(processor.tokenizer.unk_token_id),\n            "delimiter_id": EXPECTED_DELIMITER_ID,\n            "vocabulary_size": EXPECTED_VOCAB_SIZE,\n            "ctc_head_out_features": int(model.ctc_head.out_features),\n            "feature_sampling_rate": int(processor.feature_extractor.sampling_rate),\n            "feature_do_normalize": bool(processor.feature_extractor.do_normalize),\n            "tensor_path": "MMS encoder -> optional MoE -> ctc_head(73) -> logits -> log_softmax -> CTCLoss(blank=0)",\n        },\n        "label_audit": {\n            "valid_target_tokens": label_total,\n            "blank_id_0_count": int(label_counts[EXPECTED_BLANK_ID]),\n            "delimiter_id_2_count": int(label_counts[EXPECTED_DELIMITER_ID]),\n            "unknown_id_1_count": int(label_counts[1]),\n            "blank_fraction": label_counts[EXPECTED_BLANK_ID] / max(1, label_total),\n            "delimiter_fraction": label_counts[EXPECTED_DELIMITER_ID] / max(1, label_total),\n            "unknown_fraction": label_counts[1] / max(1, label_total),\n            "top_20_target_ids": label_top,\n            "decoded_examples": label_examples,\n            "decode_mismatches": label_decode_mismatches,\n        },\n        "length_audit": {\n            "sample_count": len(rows),\n            "invalid_count": len(invalid_lengths),\n            "invalid_samples": invalid_lengths[:50],\n            "zero_infinity_would_hide_invalid_samples": bool(invalid_lengths),\n        },\n        "raw_prediction_audit": {\n            "frame_count": int(frame_total),\n            "blank_argmax_fraction": frame_counts[EXPECTED_BLANK_ID] / frame_total,\n            "delimiter_argmax_fraction": frame_counts[EXPECTED_DELIMITER_ID] / frame_total,\n            "blank_mean_probability": sum(blank_probabilities) / max(1, len(blank_probabilities)),\n            "empty_prediction_rate": sum(item["empty_prediction"] for item in predictions) / max(1, len(predictions)),\n            "top_15_frame_token_ids": [[int(index), int(count)] for index, count in frame_counts.most_common(15)],\n            "cer": cer_distance / max(1, reference_chars),\n            "wer": wer_distance / max(1, reference_words),\n            "predictions": predictions,\n        },\n        "ctc_head_bias": _bias_report(model),\n    }\n    output_dir.mkdir(parents=True, exist_ok=True)\n    stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", checkpoint.name)\n    (output_dir / f"ctc_collapse_{stem}.json").write_text(\n        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    with (output_dir / f"ctc_predictions_{stem}.csv").open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(predictions[0]))\n        writer.writeheader()\n        writer.writerows(predictions)\n    return report\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data-root", type=Path, required=True)\n    parser.add_argument("--repo-root", type=Path, required=True)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--checkpoint", type=Path, action="append", default=[])\n    parser.add_argument("--sample-count", type=int, default=100)\n    parser.add_argument("--batch-size", type=int, default=4)\n    parser.add_argument("--model-name", default="facebook/mms-300m")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--make-manifest", type=Path)\n    parser.add_argument("--manifest-count", type=int, default=32)\n    args = parser.parse_args()\n    if args.make_manifest:\n        print(json.dumps(make_manifest(args.data_root, args.make_manifest, args.manifest_count, args.seed), indent=2))\n    if not args.checkpoint:\n        if args.make_manifest:\n            return\n        parser.error("--checkpoint is required unless --make-manifest is used")\n    reports = [\n        audit_checkpoint(\n            checkpoint.resolve(),\n            args.data_root.resolve(),\n            args.repo_root.resolve(),\n            args.output_dir.resolve(),\n            args.sample_count,\n            args.batch_size,\n            args.model_name,\n            args.seed,\n        )\n        for checkpoint in args.checkpoint\n    ]\n    summary = {"checkpoints": reports}\n    (args.output_dir / "ctc_collapse_summary.json").write_text(\n        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(summary, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/kaggle_ctc_collapse_diagnostics.py': '"""Use a dataset manifest when present, otherwise use the normal file walker."""\n\nfrom __future__ import annotations\n\nimport csv\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nimport ctc_collapse_diagnostics as base\n\n\nBASE_ITER_PAIRS = base.iter_pairs\n\n\ndef _manifest_rows(root: Path, split: str):\n    manifest = root / "manifest.csv"\n    if not manifest.is_file():\n        return None\n\n    def rows():\n        with manifest.open("r", newline="", encoding="utf-8-sig") as handle:\n            for row in csv.DictReader(handle):\n                if row.get("split", "").strip().lower() != split.lower():\n                    continue\n                district = row.get("district", "").strip()\n                if district not in base.DISTRICT_TO_DIALECT:\n                    continue\n                wav_rel = row.get("relative_wav", "").replace("\\\\", "/")\n                txt_rel = row.get("relative_txt", "").replace("\\\\", "/")\n                if not wav_rel or not txt_rel:\n                    continue\n\n                wav_path = root / Path(*PurePosixPath(wav_rel).parts)\n                txt_path = root / Path(*PurePosixPath(txt_rel).parts)\n                if wav_path.is_file() and txt_path.is_file():\n                    transcript = base.normalize_text(\n                        txt_path.read_text(encoding="utf-8-sig", errors="strict")\n                    )\n                    if transcript:\n                        yield {\n                            "sample_id": row.get("sample_id") or f"{district}/{wav_path.stem}",\n                            "audio": wav_path,\n                            "transcript": transcript,\n                            "district": district,\n                            "dialect": base.DISTRICT_TO_DIALECT[district],\n                        }\n                    continue\n\n                archive_path = root / f"{split}.zip"\n                if not archive_path.is_file():\n                    continue\n                with zipfile.ZipFile(archive_path) as archive:\n                    names = {item.filename for item in archive.infolist()}\n                    if wav_rel in names and txt_rel in names:\n                        wav_name, txt_name = wav_rel, txt_rel\n                    else:\n                        wav_name = next(\n                            (name for name in names if name.endswith("/" + wav_rel)),\n                            None,\n                        )\n                        txt_name = next(\n                            (name for name in names if name.endswith("/" + txt_rel)),\n                            None,\n                        )\n                    if not wav_name or not txt_name:\n                        continue\n                    transcript = base.normalize_text(\n                        archive.read(txt_name).decode("utf-8-sig", errors="strict")\n                    )\n                    if transcript:\n                        yield {\n                            "sample_id": row.get("sample_id") or f"{district}/{wav_rel}",\n                            "audio": (archive_path, wav_name),\n                            "transcript": transcript,\n                            "district": district,\n                            "dialect": base.DISTRICT_TO_DIALECT[district],\n                        }\n\n    return rows()\n\n\ndef iter_pairs(root: Path, split: str):\n    indexed = _manifest_rows(root, split)\n    if indexed is not None:\n        yield from indexed\n    else:\n        yield from BASE_ITER_PAIRS(root, split)\n\n\nbase.iter_pairs = iter_pairs\n\n\nif __name__ == "__main__":\n    base.main()\n', 'scripts/tiny_overfit_ctc.py': '#!/usr/bin/env python\n"""Controlled 32-sample plain MMS-CTC overfit experiment.\n\nThis module intentionally has no dependency on the Bengali MoE model or on a\ntraining checkpoint.  It loads a fresh ``facebook/mms-300m`` backbone, keeps\nthe validated processor contract explicit, and records enough evidence to\ndecide the tiny overfit gate at one common checkpoint.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport logging\nimport math\nimport os\nimport platform\nimport random\nimport shutil\nimport sys\nimport time\nimport traceback\nimport unicodedata\nimport zipfile\nfrom contextlib import nullcontext\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Iterable, Iterator, Sequence\n\nimport numpy as np\n\n\nROOT = Path(__file__).resolve().parents[1]\nif str(ROOT / "src") not in sys.path:\n    sys.path.insert(0, str(ROOT / "src"))\n\nfrom asr_dialect_benchmark.tokenization.simple_tokenizer import normalize_bengali_text\n\n\nEXPECTED_VOCAB_SIZE = 73\nEXPECTED_BLANK_ID = 0\nEXPECTED_UNKNOWN_ID = 1\nEXPECTED_DELIMITER_ID = 2\nEXPECTED_DELIMITER_TOKEN = "|"\nTARGET_SAMPLE_RATE = 16_000\nGATE_CER = 0.05\nGATE_WER = 0.05\nGATE_EMPTY_RATE = 0.10\nDISTRICT_TO_DIALECT = {\n    "Alipurduar": "Kamrupi",\n    "CoochBehar": "Kamrupi",\n    "Darjeeling": "Kamrupi",\n    "Jalpaiguri": "Kamrupi",\n    "Jhargram": "Jharkhandi",\n    "PaschimMedinipur": "Jharkhandi",\n    "Purulia": "Jharkhandi",\n    "Malda": "Varendri",\n    "DakshinDinajpur": "Varendri",\n    "North24Parganas": "Rarhi",\n    "Kolkata": "Rarhi",\n}\n\nLOGGER = logging.getLogger("tiny_overfit_ctc")\n\n\n@dataclass\nclass Sample:\n    sample_id: str\n    audio_path: str\n    transcript: str\n    dialect: str = ""\n    district: str = ""\n    manually_verified: str = "YES"\n    duration_seconds: float | None = None\n    audio_sha256: str = ""\n    transcript_sha256: str = ""\n    raw_transcript: str = ""\n    normalized_transcript: str = ""\n    target_ids: tuple[int, ...] = ()\n    decoded_target: str = ""\n    raw_changed: bool = False\n    adjacent_repeat_count: int = 0\n    resolved_audio_path: str = ""\n\n\ndef utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef atomic_write_text(path: Path, text: str) -> None:\n    """Write a file atomically so an interrupted run cannot corrupt JSON."""\n\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(f".{path.name}.{os.getpid()}.tmp")\n    temporary.write_text(text, encoding="utf-8")\n    os.replace(temporary, path)\n\n\ndef atomic_json(path: Path, value: Any) -> None:\n    atomic_write_text(path, json.dumps(value, ensure_ascii=False, indent=2) + "\\n")\n\n\ndef sha256_bytes(value: bytes) -> str:\n    return hashlib.sha256(value).hexdigest()\n\n\ndef sha256_text(value: str) -> str:\n    return sha256_bytes(value.encode("utf-8"))\n\n\ndef git_commit(project_root: Path) -> str | None:\n    import subprocess\n\n    try:\n        return subprocess.check_output(\n            ["git", "-C", str(project_root), "rev-parse", "HEAD"],\n            text=True,\n            stderr=subprocess.DEVNULL,\n        ).strip()\n    except Exception:\n        return None\n\n\ndef setup_logging(output_dir: Path) -> None:\n    output_dir.mkdir(parents=True, exist_ok=True)\n    log_path = output_dir / "logs" / "tiny_overfit.log"\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    LOGGER.setLevel(logging.INFO)\n    LOGGER.handlers.clear()\n    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")\n    console = logging.StreamHandler(sys.stdout)\n    console.setFormatter(formatter)\n    file_handler = logging.FileHandler(log_path, encoding="utf-8")\n    file_handler.setFormatter(formatter)\n    LOGGER.addHandler(console)\n    LOGGER.addHandler(file_handler)\n    audit_handler = logging.FileHandler(output_dir / "logs/checkpoint_audit.log", encoding="utf-8")\n    audit_handler.setFormatter(formatter)\n    LOGGER.addHandler(audit_handler)\n\n\ndef detect_environment() -> dict[str, Any]:\n    """Return and print all laptop/GPU properties needed to reproduce a run."""\n\n    import torch\n    import transformers\n\n    cuda_available = bool(torch.cuda.is_available())\n    gpu_count = int(torch.cuda.device_count()) if cuda_available else 0\n    gpu_name = torch.cuda.get_device_name(0) if gpu_count else None\n    total_vram = None\n    free_vram = None\n    if gpu_count:\n        properties = torch.cuda.get_device_properties(0)\n        total_vram = float(properties.total_memory) / (1024**3)\n        free_bytes, _ = torch.cuda.mem_get_info(0)\n        free_vram = float(free_bytes) / (1024**3)\n    report = {\n        "created_utc": utc_now(),\n        "os": platform.platform(),\n        "python": platform.python_version(),\n        "python_executable": sys.executable,\n        "torch": torch.__version__,\n        "transformers": transformers.__version__,\n        "cuda_available": cuda_available,\n        "cuda_runtime": torch.version.cuda,\n        "gpu_count": gpu_count,\n        "gpu_name": gpu_name,\n        "total_gpu_vram_gib": total_vram,\n        "available_gpu_vram_gib_at_start": free_vram,\n        "bf16_supported": bool(cuda_available and torch.cuda.is_bf16_supported()),\n        "fp16_available": bool(cuda_available),\n        "flash_attention_or_sdpa": {\n            "flash_sdp": bool(\n                cuda_available\n                and hasattr(torch.backends.cuda, "flash_sdp_enabled")\n                and torch.backends.cuda.flash_sdp_enabled()\n            ),\n            "mem_efficient_sdp": bool(\n                cuda_available\n                and hasattr(torch.backends.cuda, "mem_efficient_sdp_enabled")\n                and torch.backends.cuda.mem_efficient_sdp_enabled()\n            ),\n            "sdpa_available": hasattr(torch.nn.functional, "scaled_dot_product_attention"),\n        },\n        "cpu_threads": int(torch.get_num_threads()),\n        "distributed_initialized": bool(\n            torch.distributed.is_available() and torch.distributed.is_initialized()\n        ),\n    }\n    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n    return report\n\n\ndef set_seed(seed: int, strict_deterministic: bool) -> None:\n    import torch\n\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    if strict_deterministic:\n        torch.use_deterministic_algorithms(True, warn_only=True)\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n\n\ndef resolve_processor_path(value: Path | None, project_root: Path) -> Path:\n    """Find a processor directory without treating any model checkpoint as one."""\n\n    candidates: list[Path] = []\n    if value is not None:\n        candidates.append(value.expanduser().resolve())\n    candidates.extend(\n        [\n            project_root / "processor",\n            project_root / "data" / "processor",\n            project_root / "tiny-overfit" / "processor",\n        ]\n    )\n    for candidate in candidates:\n        if (candidate / "preprocessor_config.json").is_file() and (\n            (candidate / "tokenizer_config.json").is_file()\n            or (candidate / "vocab.json").is_file()\n        ):\n            return candidate\n    searched = ", ".join(str(candidate) for candidate in candidates)\n    raise FileNotFoundError(\n        "Validated Bengali processor not found. Supply --processor-path pointing "\n        f"to the processor with the 73-token contract. Searched: {searched}"\n    )\n\n\ndef load_and_audit_processor(path: Path) -> tuple[Any, dict[str, Any]]:\n    """Load the existing processor and fail on any contract mismatch."""\n\n    from transformers import Wav2Vec2Processor\n\n    processor = Wav2Vec2Processor.from_pretrained(path)\n    tokenizer = processor.tokenizer\n    vocab = {str(token): int(index) for token, index in tokenizer.get_vocab().items()}\n    if len(tokenizer) != EXPECTED_VOCAB_SIZE or len(vocab) != EXPECTED_VOCAB_SIZE:\n        raise ValueError(\n            f"Validated Bengali processor must have {EXPECTED_VOCAB_SIZE} tokens; "\n            f"got len(tokenizer)={len(tokenizer)}, vocab={len(vocab)}"\n        )\n    if int(tokenizer.pad_token_id) != EXPECTED_BLANK_ID:\n        raise ValueError(f"Expected blank/pad ID 0, got {tokenizer.pad_token_id}")\n    if int(tokenizer.unk_token_id) != EXPECTED_UNKNOWN_ID:\n        raise ValueError(f"Expected unknown ID 1, got {tokenizer.unk_token_id}")\n    delimiter_id = int(tokenizer.convert_tokens_to_ids(EXPECTED_DELIMITER_TOKEN))\n    if delimiter_id != EXPECTED_DELIMITER_ID:\n        raise ValueError(f"Expected delimiter ID 2, got {delimiter_id}")\n    if getattr(tokenizer, "word_delimiter_token", None) != EXPECTED_DELIMITER_TOKEN:\n        raise ValueError(\n            "Expected tokenizer.word_delimiter_token to be \'|\'; "\n            f"got {getattr(tokenizer, \'word_delimiter_token\', None)!r}"\n        )\n    feature = processor.feature_extractor\n    if int(feature.sampling_rate) != TARGET_SAMPLE_RATE:\n        raise ValueError(\n            f"Expected processor sampling rate {TARGET_SAMPLE_RATE}, got {feature.sampling_rate}"\n        )\n    if not bool(getattr(feature, "do_normalize", False)):\n        raise ValueError("Validated MMS processor must enable waveform normalization")\n    ordered_vocab = sorted(vocab.items(), key=lambda pair: pair[1])\n    if [index for _, index in ordered_vocab] != list(range(EXPECTED_VOCAB_SIZE)):\n        raise ValueError("Processor vocabulary IDs are not contiguous 0..72")\n    vocabulary_hash = sha256_text(\n        json.dumps(ordered_vocab, ensure_ascii=False, separators=(",", ":"))\n    )\n    special_tokens = {\n        name: getattr(tokenizer, name, None)\n        for name in (\n            "pad_token",\n            "unk_token",\n            "word_delimiter_token",\n            "bos_token",\n            "eos_token",\n        )\n    }\n    audit = {\n        "processor_path": str(path),\n        "processor_files": sorted(\n            str(file.relative_to(path)) for file in path.rglob("*") if file.is_file()\n        ),\n        "vocabulary_size": EXPECTED_VOCAB_SIZE,\n        "vocabulary_sha256": vocabulary_hash,\n        "token_to_id": {token: index for token, index in ordered_vocab},\n        "id_to_token": {str(index): token for token, index in ordered_vocab},\n        "special_tokens": special_tokens,\n        "blank_id": EXPECTED_BLANK_ID,\n        "unknown_id": EXPECTED_UNKNOWN_ID,\n        "delimiter_token": EXPECTED_DELIMITER_TOKEN,\n        "delimiter_id": EXPECTED_DELIMITER_ID,\n        "sampling_rate": int(feature.sampling_rate),\n        "do_normalize": bool(feature.do_normalize),\n        "normalization": {\n            "function": "asr_dialect_benchmark.tokenization.simple_tokenizer.normalize_bengali_text",\n            "nfc": True,\n            "zero_width_removed": True,\n            "bengali_codepoints_and_spaces_only": True,\n        },\n    }\n    return processor, audit\n\n\ndef split_archive_path(value: str) -> tuple[Path, str] | None:\n    if "::" not in value:\n        return None\n    archive, member = value.split("::", 1)\n    return Path(archive).expanduser(), member\n\n\ndef resolve_audio_value(value: str, manifest_path: Path, project_root: Path) -> str:\n    archive_value = split_archive_path(value)\n    if archive_value is not None:\n        archive, member = archive_value\n        if not archive.is_absolute():\n            archive = (manifest_path.parent / archive).resolve()\n        return f"{archive}::{member}"\n    path = Path(value).expanduser()\n    if path.is_absolute():\n        return str(path.resolve())\n    candidates = [(manifest_path.parent / path).resolve(), (project_root / path).resolve()]\n    for candidate in candidates:\n        if candidate.is_file():\n            return str(candidate)\n    return str(candidates[0])\n\n\ndef source_bytes(source: str) -> bytes:\n    archive_value = split_archive_path(source)\n    if archive_value is None:\n        path = Path(source)\n        if not path.is_file():\n            raise FileNotFoundError(f"Audio file does not exist: {path}")\n        return path.read_bytes()\n    archive_path, member = archive_value\n    if not archive_path.is_file():\n        raise FileNotFoundError(f"Audio archive does not exist: {archive_path}")\n    with zipfile.ZipFile(archive_path) as archive:\n        try:\n            return archive.read(member)\n        except KeyError as exc:\n            raise FileNotFoundError(f"Audio member does not exist: {archive_path}::{member}") from exc\n\n\ndef resample_audio(audio: np.ndarray, source_rate: int, target_rate: int) -> np.ndarray:\n    if source_rate == target_rate:\n        return audio.astype(np.float32, copy=False)\n    try:\n        from scipy.signal import resample_poly\n\n        gcd = math.gcd(int(source_rate), int(target_rate))\n        return resample_poly(\n            audio, target_rate // gcd, source_rate // gcd\n        ).astype(np.float32, copy=False)\n    except Exception:\n        # This fallback is only for environments without scipy; it does not\n        # normalize or clip the waveform.\n        old_x = np.linspace(0.0, 1.0, num=len(audio), endpoint=False)\n        new_length = max(1, round(len(audio) * target_rate / source_rate))\n        new_x = np.linspace(0.0, 1.0, num=new_length, endpoint=False)\n        return np.interp(new_x, old_x, audio).astype(np.float32, copy=False)\n\n\ndef read_audio_source(source: str) -> tuple[np.ndarray, int]:\n    import soundfile as sf\n\n    archive_value = split_archive_path(source)\n    if archive_value is None:\n        handle: Any = source\n    else:\n        handle = io.BytesIO(source_bytes(source))\n    audio, sample_rate = sf.read(handle, dtype="float32", always_2d=True)\n    if audio.ndim != 2 or audio.shape[1] == 0:\n        raise ValueError("decoded audio has no channels")\n    audio = audio.mean(axis=1).astype(np.float32, copy=False)\n    if len(audio) == 0 or not np.isfinite(audio).all():\n        raise ValueError("audio is empty or contains non-finite values")\n    audio = resample_audio(audio, int(sample_rate), TARGET_SAMPLE_RATE)\n    if len(audio) == 0 or not np.isfinite(audio).all():\n        raise ValueError("resampled audio is empty or non-finite")\n    if float(np.sqrt(np.mean(np.square(audio), dtype=np.float64))) <= 1e-6:\n        raise ValueError("audio is effectively silent")\n    return audio, TARGET_SAMPLE_RATE\n\n\ndef tokenizer_ids(tokenizer: Any, text: str) -> list[int]:\n    encoded = tokenizer(text, add_special_tokens=False)\n    ids = encoded["input_ids"] if isinstance(encoded, dict) else encoded.input_ids\n    if ids and isinstance(ids[0], list):\n        ids = ids[0]\n    return [int(index) for index in ids]\n\n\ndef decode_target(tokenizer: Any, ids: Sequence[int]) -> str:\n    return tokenizer.decode(\n        list(ids), group_tokens=False, skip_special_tokens=True\n    ).strip()\n\n\ndef parse_manifest(path: Path, project_root: Path, manually_verified: bool) -> list[Sample]:\n    with path.open(newline="", encoding="utf-8-sig") as handle:\n        rows = list(csv.DictReader(handle))\n    if len(rows) != 32:\n        raise ValueError(f"Tiny manifest must contain exactly 32 rows; got {len(rows)}")\n    required = {"sample_id", "transcript"}\n    if not required.issubset(rows[0] if rows else {}):\n        raise ValueError("Manifest must contain sample_id and transcript columns")\n    audio_column = "audio_path" if "audio_path" in rows[0] else "audio"\n    if audio_column not in rows[0]:\n        raise ValueError("Manifest must contain audio_path (or legacy audio) column")\n    if not manually_verified:\n        raise RuntimeError(\n            "Pass --manually-verified only after listening to all 32 pairs and checking transcripts"\n        )\n    samples: list[Sample] = []\n    seen: set[str] = set()\n    for row_number, row in enumerate(rows, start=2):\n        sample_id = str(row.get("sample_id", "")).strip()\n        if not sample_id or sample_id in seen:\n            raise ValueError(f"Manifest row {row_number} has a missing or duplicate sample_id")\n        seen.add(sample_id)\n        verified = str(row.get("manually_verified", "NO")).strip().upper()\n        if verified != "YES":\n            raise ValueError(f"{sample_id}: manually_verified must be YES, got {verified!r}")\n        raw = str(row.get("transcript", ""))\n        normalized = normalize_bengali_text(raw)\n        if not normalized:\n            raise ValueError(f"{sample_id}: transcript is empty after normalization")\n        audio = resolve_audio_value(str(row.get(audio_column, "")).strip(), path, project_root)\n        district = str(row.get("district", "")).strip()\n        dialect = str(row.get("dialect", "")).strip()\n        expected = DISTRICT_TO_DIALECT.get(district, "")\n        if district and expected and dialect and dialect.casefold() != expected.casefold():\n            raise ValueError(\n                f"{sample_id}: dialect {dialect!r} disagrees with district {district!r} ({expected})"\n            )\n        duration = row.get("duration_seconds") or row.get("duration") or ""\n        samples.append(\n            Sample(\n                sample_id=sample_id,\n                audio_path=audio,\n                transcript=normalized,\n                dialect=dialect or expected,\n                district=district,\n                manually_verified=verified,\n                duration_seconds=float(duration) if duration else None,\n                raw_transcript=raw,\n                normalized_transcript=normalized,\n                raw_changed=raw != normalized,\n                resolved_audio_path=audio,\n            )\n        )\n    return samples\n\n\ndef audit_samples(\n    samples: list[Sample],\n    processor: Any,\n    processor_audit: dict[str, Any],\n    args: argparse.Namespace,\n    output_dir: Path,\n) -> tuple[list[Sample], dict[str, Any]]:\n    tokenizer = processor.tokenizer\n    target_rows: list[dict[str, Any]] = []\n    duration_values: list[float] = []\n    character_values: list[int] = []\n    word_values: list[int] = []\n    for sample in samples:\n        try:\n            raw_bytes = source_bytes(sample.audio_path)\n            audio, sample_rate = read_audio_source(sample.audio_path)\n            duration = len(audio) / sample_rate\n            if args.max_audio_seconds is not None and duration > args.max_audio_seconds:\n                raise ValueError(\n                    f"duration {duration:.3f}s exceeds --max-audio-seconds {args.max_audio_seconds:.3f}; "\n                    "the script never crops an audio/transcript pair"\n                )\n            if sample.duration_seconds is not None and abs(sample.duration_seconds - duration) > 0.25:\n                raise ValueError(\n                    f"manifest duration {sample.duration_seconds:.3f}s disagrees with decoded {duration:.3f}s"\n                )\n            ids = tokenizer_ids(tokenizer, sample.transcript)\n            if not ids:\n                raise ValueError("encoded target is empty")\n            if any(index in {EXPECTED_BLANK_ID, EXPECTED_UNKNOWN_ID, -100} for index in ids):\n                raise ValueError(f"encoded target contains blank, unknown, or padding ID: {ids}")\n            decoded = decode_target(tokenizer, ids)\n            if normalize_bengali_text(decoded) != sample.transcript:\n                raise ValueError(\n                    f"target round-trip mismatch: normalized={sample.transcript!r}, decoded={decoded!r}"\n                )\n            repeat_count = int(sum(left == right for left, right in zip(ids, ids[1:])))\n            sample.duration_seconds = float(duration)\n            sample.audio_sha256 = sha256_bytes(raw_bytes)\n            sample.transcript_sha256 = sha256_text(sample.transcript)\n            sample.target_ids = tuple(ids)\n            sample.adjacent_repeat_count = repeat_count\n            sample.decoded_target = decoded\n            duration_values.append(duration)\n            character_values.append(len(sample.transcript.replace(" ", "")))\n            word_values.append(len(sample.transcript.split()))\n            target_rows.append(\n                {\n                    "sample_id": sample.sample_id,\n                    "audio_path": sample.audio_path,\n                    "raw_transcript": sample.raw_transcript,\n                    "normalized_transcript": sample.transcript,\n                    "encoded_token_ids": json.dumps(ids),\n                    "decoded_target": decoded,\n                    "raw_changed": sample.raw_changed,\n                    "duration_seconds": duration,\n                    "audio_sha256": sample.audio_sha256,\n                    "transcript_sha256": sample.transcript_sha256,\n                    "adjacent_repeat_count": repeat_count,\n                }\n            )\n        except Exception as exc:\n            raise ValueError(f"{sample.sample_id} ({sample.audio_path}): {exc}") from exc\n    target_audit = output_dir / "tiny_target_audit.csv"\n    with target_audit.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(target_rows[0]))\n        writer.writeheader()\n        writer.writerows(target_rows)\n    locked = output_dir / "tiny_32_manifest_locked.csv"\n    locked_fields = [\n        "sample_id",\n        "audio_path",\n        "transcript",\n        "dialect",\n        "district",\n        "duration_seconds",\n        "audio_sha256",\n        "transcript_sha256",\n        "manually_verified",\n    ]\n    with locked.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=locked_fields)\n        writer.writeheader()\n        for sample in samples:\n            writer.writerow({field: getattr(sample, field) for field in locked_fields})\n    manifest_hash = sha256_bytes(locked.read_bytes())\n    metadata = {\n        "created_utc": utc_now(),\n        "sample_count": len(samples),\n        "sample_ids": [sample.sample_id for sample in samples],\n        "manifest_sha256": manifest_hash,\n        "audio_sha256": {sample.sample_id: sample.audio_sha256 for sample in samples},\n        "transcript_sha256": {sample.sample_id: sample.transcript_sha256 for sample in samples},\n        "processor_vocabulary_sha256": processor_audit["vocabulary_sha256"],\n        "seed": args.seed,\n        "project_commit": git_commit(args.project_root),\n        "manual_verification_required": True,\n    }\n    atomic_json(output_dir / "tiny_32_manifest_metadata.json", metadata)\n    audit = {\n        "sample_count": len(samples),\n        "total_duration_seconds": float(sum(duration_values)),\n        "min_duration_seconds": float(min(duration_values)),\n        "max_duration_seconds": float(max(duration_values)),\n        "mean_duration_seconds": float(np.mean(duration_values)),\n        "transcript_characters_total": int(sum(character_values)),\n        "transcript_characters_min": int(min(character_values)),\n        "transcript_characters_max": int(max(character_values)),\n        "transcript_words_total": int(sum(word_values)),\n        "transcript_words_min": int(min(word_values)),\n        "transcript_words_max": int(max(word_values)),\n        "manifest_sha256": manifest_hash,\n        "rejected_samples": [],\n    }\n    atomic_json(output_dir / "tiny_data_audit.json", audit)\n    return samples, audit\n\n\ndef auto_defaults(environment: dict[str, Any], args: argparse.Namespace) -> None:\n    vram = environment.get("total_gpu_vram_gib") or 0.0\n    if args.batch_size is None:\n        args.batch_size = 1 if vram < 8 else 2\n    if args.gradient_accumulation_steps is None:\n        args.gradient_accumulation_steps = 8 if vram < 8 else 4\n    if args.trainable_encoder_layers is None:\n        args.trainable_encoder_layers = 2 if vram < 6 else 4\n    if args.gradient_checkpointing is None:\n        args.gradient_checkpointing = bool(vram < 12)\n    if args.precision is None:\n        args.precision = "fp16" if environment["cuda_available"] else "none"\n    if args.num_workers is None:\n        args.num_workers = 0 if os.name == "nt" else min(2, os.cpu_count() or 1)\n    if args.precision == "bf16" and not environment["bf16_supported"]:\n        raise RuntimeError("--bf16 requested but this GPU does not support BF16")\n    if args.precision == "fp16" and not environment["fp16_available"]:\n        raise RuntimeError("--fp16 requested but CUDA/FP16 is unavailable")\n    if not environment["cuda_available"]:\n        args.precision = "none"\n\n\ndef build_model(args: argparse.Namespace, device: Any) -> tuple[Any, dict[str, Any]]:\n    """Load only the public MMS backbone and initialize a fresh 73-way head."""\n\n    import torch\n    from transformers import Wav2Vec2ForCTC\n\n    if args.model_name != "facebook/mms-300m":\n        LOGGER.warning("Model source override: %s", args.model_name)\n    model = Wav2Vec2ForCTC.from_pretrained(\n        args.model_name,\n        ignore_mismatched_sizes=True,\n        vocab_size=EXPECTED_VOCAB_SIZE,\n        pad_token_id=EXPECTED_BLANK_ID,\n    )\n    hidden_size = int(model.config.hidden_size)\n    # Never reuse a downloaded CTC head.  Only the MMS acoustic backbone is\n    # loaded; this linear layer is freshly initialized for the validated vocab.\n    model.lm_head = torch.nn.Linear(hidden_size, EXPECTED_VOCAB_SIZE)\n    model.config.vocab_size = EXPECTED_VOCAB_SIZE\n    model.config.pad_token_id = EXPECTED_BLANK_ID\n    model.config.ctc_loss_reduction = "mean"\n    model.config.ctc_zero_infinity = False\n    model.config.use_cache = False\n    for parameter in model.parameters():\n        parameter.requires_grad = False\n    for parameter in model.lm_head.parameters():\n        parameter.requires_grad = True\n    wav2vec = model.wav2vec2\n    feature_extractor = getattr(wav2vec, "feature_extractor", None)\n    if feature_extractor is None:\n        raise RuntimeError("MMS backbone does not expose a convolutional feature encoder")\n    for parameter in feature_extractor.parameters():\n        parameter.requires_grad = False\n    layers = getattr(getattr(wav2vec, "encoder", None), "layers", None)\n    if layers is None:\n        raise RuntimeError("MMS backbone does not expose transformer encoder layers")\n    layer_count = len(layers)\n    requested = int(args.trainable_encoder_layers)\n    if requested < 1 or requested > layer_count:\n        raise ValueError(f"--trainable-encoder-layers must be 1..{layer_count}, got {requested}")\n    for layer in layers[-requested:]:\n        for parameter in layer.parameters():\n            parameter.requires_grad = True\n    if args.gradient_checkpointing:\n        model.gradient_checkpointing_enable()\n    model.to(device)\n    if any("moe" in name.lower() or "router" in name.lower() or "expert" in name.lower() for name, _ in model.named_modules()):\n        raise AssertionError("Plain MMS model unexpectedly contains an MoE/router/expert module")\n    trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]\n    if not any(name.startswith("lm_head.") for name in trainable_names):\n        raise AssertionError("Fresh CTC head is not trainable")\n    if not any("encoder.layers" in name for name in trainable_names):\n        raise AssertionError("No transformer encoder layer is trainable")\n    feature_count = sum(p.numel() for p in feature_extractor.parameters())\n    encoder_count = sum(p.numel() for p in wav2vec.encoder.parameters())\n    head_count = sum(p.numel() for p in model.lm_head.parameters())\n    total_count = sum(p.numel() for p in model.parameters())\n    trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    audit = {\n        "model_source": args.model_name,\n        "output_vocabulary_size": EXPECTED_VOCAB_SIZE,\n        "blank_id": EXPECTED_BLANK_ID,\n        "unknown_id": EXPECTED_UNKNOWN_ID,\n        "delimiter_id": EXPECTED_DELIMITER_ID,\n        "total_parameters": int(total_count),\n        "trainable_parameters": int(trainable_count),\n        "frozen_parameters": int(total_count - trainable_count),\n        "feature_encoder_parameters": int(feature_count),\n        "transformer_encoder_parameters": int(encoder_count),\n        "ctc_head_parameters": int(head_count),\n        "transformer_layer_count": layer_count,\n        "trainable_transformer_layers": list(range(layer_count - requested, layer_count)),\n        "trainable_parameter_names": trainable_names,\n        "moe_components_present": False,\n        "dialect_classifier_present": False,\n        "augmentation_present": False,\n        "used_failed_checkpoint": False,\n    }\n    atomic_json(args.output_dir / "model_audit.json", audit)\n    LOGGER.info(\n        "Model: %s total=%d trainable=%d frozen=%d; trainable layers=%s",\n        args.model_name,\n        total_count,\n        trainable_count,\n        total_count - trainable_count,\n        audit["trainable_transformer_layers"],\n    )\n    return model, audit\n\n\ndef feature_output_lengths(model: Any, input_lengths: Any) -> Any:\n    encoder = getattr(model, "wav2vec2", model)\n    method = getattr(encoder, "_get_feat_extract_output_lengths", None)\n    if method is None:\n        method = getattr(model, "_get_feat_extract_output_lengths", None)\n    if method is None:\n        raise RuntimeError("MMS model does not provide official feature output lengths")\n    return method(input_lengths).long()\n\n\ndef make_batch(\n    indices: Sequence[int],\n    samples: list[Sample],\n    processor: Any,\n    device: Any,\n) -> dict[str, Any]:\n    import torch\n    from torch.nn.utils.rnn import pad_sequence\n\n    arrays = []\n    raw_lengths = []\n    for index in indices:\n        audio, rate = read_audio_source(samples[index].audio_path)\n        if rate != TARGET_SAMPLE_RATE:\n            raise AssertionError("audio loader failed to return 16 kHz")\n        arrays.append(audio)\n        raw_lengths.append(len(audio))\n    processed = processor(\n        arrays,\n        sampling_rate=TARGET_SAMPLE_RATE,\n        return_tensors="pt",\n        padding=True,\n        return_attention_mask=True,\n    )\n    if "attention_mask" not in processed:\n        raise RuntimeError("validated MMS processor did not return an attention mask")\n    labels = [torch.tensor(samples[index].target_ids, dtype=torch.long) for index in indices]\n    padded = pad_sequence(labels, batch_first=True, padding_value=-100)\n    target_lengths = torch.tensor([len(sample) for sample in labels], dtype=torch.long)\n    return {\n        "input_values": processed["input_values"].to(device),\n        "attention_mask": processed["attention_mask"].to(device),\n        "input_lengths": processed["attention_mask"].sum(-1).long().to(device),\n        "targets": padded.to(device),\n        "target_lengths": target_lengths.to(device),\n        "raw_waveform_lengths": torch.tensor(raw_lengths, dtype=torch.long),\n        "sample_indices": list(indices),\n    }\n\n\ndef validate_ctc_lengths(model: Any, batch: dict[str, Any], output_time: int) -> tuple[Any, Any]:\n    import torch\n\n    output_lengths = feature_output_lengths(model, batch["input_lengths"])\n    if int(output_lengths.max().item()) > output_time:\n        raise RuntimeError(\n            f"Official CTC lengths exceed logits time dimension: {output_lengths.tolist()} > {output_time}"\n        )\n    flat_targets: list[Any] = []\n    required: list[int] = []\n    offset = 0\n    for row, target_length in enumerate(batch["target_lengths"].tolist()):\n        target = batch["targets"][row, :target_length]\n        if (target == EXPECTED_BLANK_ID).any() or (target == EXPECTED_UNKNOWN_ID).any() or (target == -100).any():\n            raise RuntimeError(f"Invalid target IDs for batch sample index {batch[\'sample_indices\'][row]}")\n        repeats = int((target[1:] == target[:-1]).sum().item())\n        required.append(int(target_length) + repeats)\n        flat_targets.append(target)\n        offset += int(target_length)\n    flat = torch.cat(flat_targets).long() if flat_targets else torch.empty(0, dtype=torch.long, device=batch["targets"].device)\n    required_tensor = torch.tensor(required, dtype=torch.long, device=output_lengths.device)\n    if (output_lengths < required_tensor).any():\n        details = [\n            {\n                "sample_index": batch["sample_indices"][index],\n                "input_length": int(output_lengths[index].item()),\n                "required_length": int(required_tensor[index].item()),\n                "target_length": int(batch["target_lengths"][index].item()),\n            }\n            for index in range(len(required))\n            if output_lengths[index] < required_tensor[index]\n        ]\n        raise RuntimeError(f"CTC repeated-label length constraint failed: {details}")\n    return flat, output_lengths\n\n\ndef ctc_forward(model: Any, batch: dict[str, Any], autocast_context: Any) -> tuple[Any, Any, Any, Any]:\n    import torch\n    import torch.nn.functional as F\n\n    with autocast_context:\n        outputs = model(\n            input_values=batch["input_values"],\n            attention_mask=batch["attention_mask"],\n        )\n        logits = outputs.logits\n    logits_float = logits.float()\n    if logits_float.ndim != 3 or logits_float.shape[-1] != EXPECTED_VOCAB_SIZE:\n        raise RuntimeError(f"Expected logits [B,T,73], got {tuple(logits_float.shape)}")\n    flat_targets, output_lengths = validate_ctc_lengths(model, batch, logits_float.shape[1])\n    loss = F.ctc_loss(\n        logits_float.log_softmax(-1).transpose(0, 1),\n        flat_targets,\n        output_lengths,\n        batch["target_lengths"],\n        blank=EXPECTED_BLANK_ID,\n        zero_infinity=False,\n        reduction="mean",\n    )\n    if not torch.isfinite(loss).item() or not torch.isfinite(logits_float).all().item():\n        raise FloatingPointError("CTC loss or logits became non-finite")\n    return loss, logits_float, output_lengths, flat_targets\n\n\ndef collapse_ctc(ids: Iterable[int]) -> list[int]:\n    collapsed: list[int] = []\n    previous: int | None = None\n    for raw in ids:\n        token_id = int(raw)\n        if token_id == previous:\n            continue\n        previous = token_id\n        if token_id != EXPECTED_BLANK_ID:\n            collapsed.append(token_id)\n    return collapsed\n\n\ndef edit_distance(reference: Sequence[Any], hypothesis: Sequence[Any]) -> int:\n    previous = list(range(len(hypothesis) + 1))\n    for index, ref in enumerate(reference, start=1):\n        current = [index]\n        for other, hyp in enumerate(hypothesis, start=1):\n            current.append(\n                min(\n                    current[-1] + 1,\n                    previous[other] + 1,\n                    previous[other - 1] + (ref != hyp),\n                )\n            )\n        previous = current\n    return previous[-1]\n\n\ndef grad_norm(parameters: Iterable[Any]) -> float:\n    import torch\n\n    total = 0.0\n    for parameter in parameters:\n        if parameter.grad is None:\n            continue\n        values = parameter.grad.detach().float()\n        if not torch.isfinite(values).all():\n            return float("nan")\n        total += float(values.pow(2).sum().item())\n    return math.sqrt(total)\n\n\ndef parameter_changed(before: dict[str, Any], model: Any) -> bool:\n    for name, old in before.items():\n        current = dict(model.named_parameters())[name].detach()\n        if not bool((current != old).any().item()):\n            continue\n        return True\n    return False\n\n\ndef prediction_metrics(\n    processor: Any,\n    samples: list[Sample],\n    predictions: list[dict[str, Any]],\n    frame_count: int,\n    counts: dict[int, int],\n    blank_probability_sum: float,\n    delimiter_probability_sum: float,\n    entropy_sum: float,\n    loss: float,\n) -> dict[str, Any]:\n    references = [sample.transcript for sample in samples]\n    character_denominator = sum(len(ref.replace(" ", "")) for ref in references)\n    word_denominator = sum(len(ref.split()) for ref in references)\n    character_errors = sum(\n        edit_distance(list(ref.replace(" ", "")), list(row["prediction"].replace(" ", "")))\n        for ref, row in zip(references, predictions)\n    )\n    word_errors = sum(\n        edit_distance(ref.split(), row["prediction"].split())\n        for ref, row in zip(references, predictions)\n    )\n    blank_fraction = counts.get(EXPECTED_BLANK_ID, 0) / max(1, frame_count)\n    delimiter_fraction = counts.get(EXPECTED_DELIMITER_ID, 0) / max(1, frame_count)\n    return {\n        "eval_ctc_loss": float(loss),\n        "cer": character_errors / max(1, character_denominator),\n        "wer": word_errors / max(1, word_denominator),\n        "empty_prediction_rate": sum(not row["prediction"] for row in predictions) / max(1, len(predictions)),\n        "mean_prediction_length_chars": float(np.mean([len(row["prediction"].replace(" ", "")) for row in predictions])),\n        "mean_prediction_length_words": float(np.mean([len(row["prediction"].split()) for row in predictions])),\n        "blank_argmax_fraction": blank_fraction,\n        "delimiter_argmax_fraction": delimiter_fraction,\n        "unknown_argmax_fraction": counts.get(EXPECTED_UNKNOWN_ID, 0) / max(1, frame_count),\n        "mean_blank_probability": blank_probability_sum / max(1, frame_count),\n        "mean_delimiter_probability": delimiter_probability_sum / max(1, frame_count),\n        "frame_entropy": entropy_sum / max(1, frame_count),\n        "valid_frame_count": int(frame_count),\n        "raw_argmax_distribution": {str(key): int(value) for key, value in sorted(counts.items())},\n        "predictions": predictions,\n    }\n\n\ndef evaluate(\n    model: Any,\n    samples: list[Sample],\n    processor: Any,\n    device: Any,\n    batch_size: int,\n    precision: str,\n) -> dict[str, Any]:\n    import torch\n\n    model.eval()\n    autocast_dtype = torch.float16 if precision == "fp16" else torch.bfloat16\n    use_autocast = device.type == "cuda" and precision in {"fp16", "bf16"}\n    predictions: list[dict[str, Any]] = []\n    counts: dict[int, int] = {}\n    blank_probability_sum = 0.0\n    delimiter_probability_sum = 0.0\n    entropy_sum = 0.0\n    frame_count = 0\n    loss_sum = 0.0\n    trace_rows: list[dict[str, Any]] = []\n    length_rows: list[dict[str, Any]] = []\n    with torch.inference_mode():\n        for start in range(0, len(samples), batch_size):\n            indices = list(range(start, min(len(samples), start + batch_size)))\n            batch = make_batch(indices, samples, processor, device)\n            context = torch.autocast("cuda", dtype=autocast_dtype) if use_autocast else nullcontext()\n            loss, logits, output_lengths, _ = ctc_forward(model, batch, context)\n            loss_sum += float(loss.item()) * len(indices)\n            probabilities = logits.softmax(-1)\n            frame_ids = logits.argmax(-1)\n            for local, sample_index in enumerate(indices):\n                length = int(output_lengths[local].item())\n                raw_ids = frame_ids[local, :length].detach().cpu().tolist()\n                raw_probs = probabilities[local, :length]\n                for token_id in raw_ids:\n                    counts[token_id] = counts.get(token_id, 0) + 1\n                frame_count += length\n                blank_probability_sum += float(raw_probs[:, EXPECTED_BLANK_ID].sum().item())\n                delimiter_probability_sum += float(raw_probs[:, EXPECTED_DELIMITER_ID].sum().item())\n                entropy_sum += float((-(raw_probs * raw_probs.clamp_min(1e-12).log()).sum(-1)).sum().item())\n                collapsed = collapse_ctc(raw_ids)\n                prediction = processor.tokenizer.decode(\n                    collapsed, group_tokens=False, skip_special_tokens=True\n                ).strip()\n                sample = samples[sample_index]\n                length_rows.append(\n                    {\n                        "sample_id": sample.sample_id,\n                        "raw_waveform_length": int(batch["raw_waveform_lengths"][local].item()),\n                        "attention_mask_length": int(batch["input_lengths"][local].item()),\n                        "ctc_logit_length": length,\n                        "target_length": len(sample.target_ids),\n                        "adjacent_repeat_count": sample.adjacent_repeat_count,\n                    }\n                )\n                reference = sample.transcript\n                row = {\n                    "step": None,\n                    "sample_id": sample.sample_id,\n                    "audio_path": sample.audio_path,\n                    "dialect": sample.dialect,\n                    "district": sample.district,\n                    "reference": reference,\n                    "prediction": prediction,\n                    "reference_length_chars": len(reference.replace(" ", "")),\n                    "prediction_length_chars": len(prediction.replace(" ", "")),\n                    "reference_length_words": len(reference.split()),\n                    "prediction_length_words": len(prediction.split()),\n                    "cer": edit_distance(list(reference.replace(" ", "")), list(prediction.replace(" ", ""))) / max(1, len(reference.replace(" ", ""))),\n                    "wer": edit_distance(reference.split(), prediction.split()) / max(1, len(reference.split())),\n                    "empty_prediction": not prediction,\n                    "raw_argmax_token_count": len(raw_ids),\n                    "raw_blank_fraction": raw_ids.count(EXPECTED_BLANK_ID) / max(1, len(raw_ids)),\n                    "raw_delimiter_fraction": raw_ids.count(EXPECTED_DELIMITER_ID) / max(1, len(raw_ids)),\n                    "mean_blank_probability": float(raw_probs[:, EXPECTED_BLANK_ID].mean().item()),\n                    "mean_delimiter_probability": float(raw_probs[:, EXPECTED_DELIMITER_ID].mean().item()),\n                }\n                predictions.append(row)\n                if len(trace_rows) < 4:\n                    trace_rows.append(\n                        {\n                            "sample_id": sample.sample_id,\n                            "raw_argmax_ids": raw_ids,\n                            "collapsed_ids": collapsed,\n                        }\n                    )\n    model.train()\n    metrics = prediction_metrics(\n        processor,\n        samples,\n        predictions,\n        frame_count,\n        counts,\n        blank_probability_sum,\n        delimiter_probability_sum,\n        entropy_sum,\n        loss_sum / len(samples),\n    )\n    metrics["predictions"] = predictions\n    metrics["raw_traces"] = trace_rows\n    metrics["ctc_length_rows"] = length_rows\n    return metrics\n\n\ndef write_predictions(output_dir: Path, step: int, predictions: list[dict[str, Any]], trace_rows: list[dict[str, Any]]) -> None:\n    prediction_dir = output_dir / "predictions"\n    trace_dir = output_dir / "token_traces"\n    prediction_dir.mkdir(parents=True, exist_ok=True)\n    trace_dir.mkdir(parents=True, exist_ok=True)\n    fields = list(predictions[0]) if predictions else []\n    snapshot = prediction_dir / f"ctc_predictions_step_{step:06d}.csv"\n    with snapshot.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(predictions)\n    shutil.copyfile(snapshot, output_dir / "ctc_predictions_latest.csv")\n    atomic_json(trace_dir / f"raw_tokens_step_{step:06d}.json", trace_rows)\n\n\ndef optimizer_setup(model: Any, args: argparse.Namespace) -> tuple[Any, Any, list[Any], list[Any]]:\n    import torch\n    from torch.optim.lr_scheduler import LambdaLR\n\n    head_decay, head_no_decay, encoder_decay, encoder_no_decay = [], [], [], []\n    head_parameters, encoder_parameters = [], []\n    for name, parameter in model.named_parameters():\n        if not parameter.requires_grad:\n            continue\n        is_no_decay = name.endswith("bias") or "layer_norm" in name.lower() or "layernorm" in name.lower()\n        if name.startswith("lm_head."):\n            head_parameters.append(parameter)\n            (head_no_decay if is_no_decay else head_decay).append(parameter)\n        else:\n            encoder_parameters.append(parameter)\n            (encoder_no_decay if is_no_decay else encoder_decay).append(parameter)\n    groups = [\n        {"params": head_decay, "lr": args.head_lr, "weight_decay": args.weight_decay, "group_name": "ctc_head_decay"},\n        {"params": head_no_decay, "lr": args.head_lr, "weight_decay": 0.0, "group_name": "ctc_head_no_decay"},\n        {"params": encoder_decay, "lr": args.encoder_lr, "weight_decay": args.weight_decay, "group_name": "encoder_decay"},\n        {"params": encoder_no_decay, "lr": args.encoder_lr, "weight_decay": 0.0, "group_name": "encoder_no_decay"},\n    ]\n    groups = [group for group in groups if group["params"]]\n    optimizer = torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8)\n    warmup = max(0, int(args.warmup_steps))\n\n    def schedule(step: int) -> float:\n        if warmup <= 0:\n            return 1.0\n        return min(1.0, float(step + 1) / warmup)\n\n    scheduler = LambdaLR(optimizer, schedule)\n    return optimizer, scheduler, head_parameters, encoder_parameters\n\n\ndef save_checkpoint(\n    path: Path,\n    model: Any,\n    optimizer: Any,\n    scheduler: Any,\n    scaler: Any,\n    step: int,\n    best_metrics: dict[str, Any] | None,\n    args: argparse.Namespace,\n    processor_audit: dict[str, Any],\n    manifest_hash: str,\n    environment: dict[str, Any],\n) -> None:\n    import torch\n\n    payload = {\n        "model_state_dict": model.state_dict(),\n        "optimizer_state_dict": optimizer.state_dict(),\n        "scheduler_state_dict": scheduler.state_dict(),\n        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,\n        "global_optimizer_step": int(step),\n        "epoch": int(step),\n        "best_metrics": best_metrics,\n        "processor_path": processor_audit["processor_path"],\n        "processor_vocabulary_sha256": processor_audit["vocabulary_sha256"],\n        "manifest_sha256": manifest_hash,\n        "training_arguments": {key: str(value) if isinstance(value, Path) else value for key, value in vars(args).items()},\n        "random_seed": args.seed,\n        "environment": environment,\n        "model_source": args.model_name,\n        "used_failed_checkpoint": False,\n        "moe_components_present": False,\n        "dialect_loss_present": False,\n    }\n    temporary = path.with_name(f".{path.name}.{os.getpid()}.tmp")\n    torch.save(payload, temporary)\n    os.replace(temporary, path)\n\n\ndef gate_passed(metrics: dict[str, Any]) -> bool:\n    return bool(\n        metrics["cer"] <= GATE_CER\n        and metrics["wer"] <= GATE_WER\n        and metrics["empty_prediction_rate"] < GATE_EMPTY_RATE\n    )\n\n\ndef status_payload(\n    status: str,\n    args: argparse.Namespace,\n    latest_step: int,\n    latest_metrics: dict[str, Any],\n    best_metrics: dict[str, Any] | None,\n    gate_metrics: dict[str, Any] | None,\n    reason: str | None = None,\n) -> dict[str, Any]:\n    return {\n        "status": status,\n        "passed": bool(gate_metrics),\n        "reason": reason,\n        "best_step": best_metrics.get("step") if best_metrics else None,\n        "best_checkpoint": str(args.output_dir / "tiny_overfit_best.pt") if best_metrics else None,\n        "best_cer": best_metrics.get("cer") if best_metrics else None,\n        "best_wer": best_metrics.get("wer") if best_metrics else None,\n        "best_empty_prediction_rate": best_metrics.get("empty_prediction_rate") if best_metrics else None,\n        "best_same_checkpoint_gate_metrics": {\n            "step": gate_metrics.get("step") if gate_metrics else None,\n            "cer": gate_metrics.get("cer") if gate_metrics else None,\n            "wer": gate_metrics.get("wer") if gate_metrics else None,\n            "empty_prediction_rate": gate_metrics.get("empty_prediction_rate") if gate_metrics else None,\n        },\n        "latest_step": int(latest_step),\n        "latest_metrics": latest_metrics,\n        "processor_vocabulary_size": EXPECTED_VOCAB_SIZE,\n        "blank_id": EXPECTED_BLANK_ID,\n        "unknown_id": EXPECTED_UNKNOWN_ID,\n        "delimiter_id": EXPECTED_DELIMITER_ID,\n        "model_source": args.model_name,\n        "used_moe_checkpoint": False,\n        "moe_components_present": False,\n        "dialect_loss_present": False,\n        "augmentation_present": False,\n    }\n\n\ndef json_args(args: argparse.Namespace) -> dict[str, Any]:\n    return {key: str(value) if isinstance(value, Path) else value for key, value in vars(args).items()}\n\n\ndef run_experiment(args: argparse.Namespace) -> int:\n    import torch\n\n    args.project_root = args.project_root.resolve()\n    args.output_dir = args.output_dir.resolve()\n    setup_logging(args.output_dir)\n    environment = detect_environment()\n    if environment["distributed_initialized"]:\n        raise RuntimeError("Distributed training is already initialized; plain test refuses to continue")\n    if not environment["cuda_available"] and not args.allow_cpu:\n        raise RuntimeError("CUDA is unavailable. Use a laptop GPU or pass --allow-cpu explicitly.")\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    auto_defaults(environment, args)\n    if environment["gpu_count"] > 1:\n        LOGGER.warning("%d GPUs are visible; using only device 0 and no DDP/DataParallel", environment["gpu_count"])\n    set_seed(args.seed, args.strict_deterministic)\n    processor_path = resolve_processor_path(args.processor_path, args.project_root)\n    processor, processor_audit = load_and_audit_processor(processor_path)\n    args.processor_path = processor_path\n    atomic_json(args.output_dir / "environment.json", {**environment, "selected_device": str(device), "precision": args.precision})\n    atomic_json(args.output_dir / "processor_audit.json", processor_audit)\n    atomic_json(args.output_dir / "run_config.json", {"created_utc": utc_now(), **json_args(args), "environment": environment})\n    samples = parse_manifest(args.manifest.resolve(), args.project_root, args.manually_verified)\n    samples, data_audit = audit_samples(samples, processor, processor_audit, args, args.output_dir)\n    model, model_audit = build_model(args, device)\n    optimizer, scheduler, head_parameters, encoder_parameters = optimizer_setup(model, args)\n    scaler = torch.cuda.amp.GradScaler(enabled=args.precision == "fp16" and device.type == "cuda")\n    manifest_hash = sha256_bytes((args.output_dir / "tiny_32_manifest_locked.csv").read_bytes())\n    history_path = args.output_dir / "tiny_overfit_history.jsonl"\n    status_path = args.output_dir / "tiny_overfit_status.json"\n    best_metrics: dict[str, Any] | None = None\n    gate_metrics: dict[str, Any] | None = None\n    latest_metrics: dict[str, Any] = {}\n    best_cer = float("inf")\n\n    def append_record(record: dict[str, Any]) -> None:\n        nonlocal best_metrics, gate_metrics, best_cer, latest_metrics\n        predictions = record.pop("predictions", [])\n        traces = record.pop("raw_traces", [])\n        length_rows = record.pop("ctc_length_rows", [])\n        for prediction in predictions:\n            prediction["step"] = int(record["step"])\n        for length_row in length_rows:\n            length_row["step"] = int(record["step"])\n        latest_metrics = dict(record)\n        with history_path.open("a", encoding="utf-8") as handle:\n            handle.write(json.dumps(record, ensure_ascii=False) + "\\n")\n        write_predictions(args.output_dir, int(record["step"]), predictions, traces)\n        with (args.output_dir / "ctc_lengths_audit.jsonl").open("a", encoding="utf-8") as length_handle:\n            for length_row in length_rows:\n                length_handle.write(json.dumps(length_row, ensure_ascii=False) + "\\n")\n        if record["cer"] < best_cer:\n            best_cer = record["cer"]\n            best_metrics = dict(record)\n            shutil.copyfile(\n                args.output_dir / "predictions" / f"ctc_predictions_step_{int(record[\'step\']):06d}.csv",\n                args.output_dir / "ctc_predictions_best.csv",\n            )\n        if gate_passed(record) and (gate_metrics is None or record["cer"] < gate_metrics["cer"]):\n            gate_metrics = dict(record)\n        atomic_json(status_path, status_payload("running", args, int(record["step"]), record, best_metrics, gate_metrics))\n        LOGGER.info(\n            "step=%d loss=%.5f CER=%.4f WER=%.4f empty=%.3f blank=%.4f delimiter=%.4f head_grad=%s encoder_grad=%s",\n            record["step"], record["train_loss"] if record.get("train_loss") is not None else float("nan"), record["cer"], record["wer"],\n            record["empty_prediction_rate"], record["blank_argmax_fraction"], record["delimiter_argmax_fraction"],\n            record.get("head_grad_norm"), record.get("encoder_grad_norm"),\n        )\n\n    def evaluate_and_record(step: int, train_loss: float, head_grad: float | None, encoder_grad: float | None, step_seconds: float = 0.0) -> None:\n        eval_start = time.perf_counter()\n        metrics = evaluate(model, samples, processor, device, args.batch_size, args.precision)\n        record = {\n            "step": int(step),\n            "train_loss": float(train_loss),\n            "head_grad_norm": head_grad,\n            "encoder_grad_norm": encoder_grad,\n            "total_grad_norm": None if head_grad is None or encoder_grad is None else float(math.sqrt(head_grad**2 + encoder_grad**2)),\n            "head_lr": float(max(group["lr"] for group in optimizer.param_groups if group.get("group_name", "").startswith("ctc_head"))),\n            "encoder_lr": float(max(group["lr"] for group in optimizer.param_groups if group.get("group_name", "").startswith("encoder"))),\n            "gpu_allocated_gib": float(torch.cuda.memory_allocated() / 1024**3) if device.type == "cuda" else 0.0,\n            "gpu_reserved_gib": float(torch.cuda.memory_reserved() / 1024**3) if device.type == "cuda" else 0.0,\n            "gpu_max_allocated_gib": float(torch.cuda.max_memory_allocated() / 1024**3) if device.type == "cuda" else 0.0,\n            "step_seconds": float(step_seconds),\n            "evaluation_seconds": float(time.perf_counter() - eval_start),\n            **metrics,\n        }\n        append_record(record)\n        atomic_json(args.output_dir / "ctc_collapse_summary.json", {key: value for key, value in record.items() if key not in {"predictions", "raw_traces"}})\n\n    initial = evaluate(model, samples, processor, device, args.batch_size, args.precision)\n    initial_record = {\n        "step": 0,\n        "train_loss": None,\n        "head_grad_norm": None,\n        "encoder_grad_norm": None,\n        "total_grad_norm": None,\n        "head_lr": float(max(group["lr"] for group in optimizer.param_groups if group.get("group_name", "").startswith("ctc_head"))),\n        "encoder_lr": float(max(group["lr"] for group in optimizer.param_groups if group.get("group_name", "").startswith("encoder"))),\n        "gpu_allocated_gib": float(torch.cuda.memory_allocated() / 1024**3) if device.type == "cuda" else 0.0,\n        "gpu_reserved_gib": float(torch.cuda.memory_reserved() / 1024**3) if device.type == "cuda" else 0.0,\n        "gpu_max_allocated_gib": float(torch.cuda.max_memory_allocated() / 1024**3) if device.type == "cuda" else 0.0,\n        "step_seconds": 0.0,\n        "evaluation_seconds": 0.0,\n        **initial,\n    }\n    append_record(initial_record)\n    save_checkpoint(args.output_dir / "tiny_overfit_last.pt", model, optimizer, scheduler, scaler, 0, best_metrics, args, processor_audit, manifest_hash, environment)\n    save_checkpoint(args.output_dir / "tiny_overfit_best.pt", model, optimizer, scheduler, scaler, 0, best_metrics, args, processor_audit, manifest_hash, environment)\n    if args.dry_run:\n        atomic_json(status_path, status_payload("dry_run", args, 0, latest_metrics, best_metrics, gate_metrics, "No optimizer update requested"))\n        return 0\n\n    requested_steps = 3 if args.smoke_test else args.max_steps\n    model.train()\n    try:\n        for step in range(1, requested_steps + 1):\n            step_start = time.perf_counter()\n            optimizer.zero_grad(set_to_none=True)\n            before = {\n                "lm_head.weight": model.lm_head.weight.detach().clone(),\n                f"wav2vec2.encoder.layers.{len(model.wav2vec2.encoder.layers)-1}.attention.k_proj.weight": dict(model.named_parameters())[f"wav2vec2.encoder.layers.{len(model.wav2vec2.encoder.layers)-1}.attention.k_proj.weight"].detach().clone(),\n            }\n            train_loss_sum = 0.0\n            head_norm = None\n            encoder_norm = None\n            for micro in range(args.gradient_accumulation_steps):\n                start = ((step - 1) * args.gradient_accumulation_steps + micro) * args.batch_size\n                indices = [(start + offset) % len(samples) for offset in range(args.batch_size)]\n                batch = make_batch(indices, samples, processor, device)\n                context = torch.autocast("cuda", dtype=torch.float16 if args.precision == "fp16" else torch.bfloat16) if device.type == "cuda" and args.precision in {"fp16", "bf16"} else nullcontext()\n                loss, logits, _, _ = ctc_forward(model, batch, context)\n                train_loss_sum += float(loss.item())\n                scaled_loss = loss / args.gradient_accumulation_steps\n                if scaler.is_enabled():\n                    scaler.scale(scaled_loss).backward()\n                else:\n                    scaled_loss.backward()\n            if scaler.is_enabled():\n                scaler.unscale_(optimizer)\n            head_norm = grad_norm(head_parameters)\n            encoder_norm = grad_norm(encoder_parameters)\n            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            if not torch.isfinite(torch.as_tensor(total_norm)).item():\n                raise FloatingPointError("gradient norm became non-finite")\n            if scaler.is_enabled():\n                scaler.step(optimizer)\n                scaler.update()\n            else:\n                optimizer.step()\n            scheduler.step()\n            if not parameter_changed(before, model):\n                raise RuntimeError("No selected model parameter changed after optimizer update")\n            step_seconds = time.perf_counter() - step_start\n            if step == 1 or step % args.eval_steps == 0 or step == requested_steps:\n                evaluate_and_record(step, train_loss_sum / args.gradient_accumulation_steps, head_norm, encoder_norm, step_seconds)\n            save_checkpoint(args.output_dir / "tiny_overfit_last.pt", model, optimizer, scheduler, scaler, step, best_metrics, args, processor_audit, manifest_hash, environment)\n            if best_metrics and int(best_metrics["step"]) == step:\n                save_checkpoint(args.output_dir / "tiny_overfit_best.pt", model, optimizer, scheduler, scaler, step, best_metrics, args, processor_audit, manifest_hash, environment)\n    except KeyboardInterrupt:\n        atomic_json(status_path, status_payload("interrupted", args, int(latest_metrics.get("step", 0)), latest_metrics, best_metrics, gate_metrics, "User interrupted the run"))\n        raise\n    except torch.cuda.OutOfMemoryError as exc:\n        if device.type == "cuda":\n            torch.cuda.empty_cache()\n        error_path = args.output_dir / "logs" / "error_traceback.txt"\n        atomic_write_text(error_path, traceback.format_exc())\n        atomic_json(status_path, status_payload("failed", args, int(latest_metrics.get("step", 0)), latest_metrics, best_metrics, gate_metrics, "CUDA out of memory; reduce batch size/layers or enable checkpointing"))\n        raise RuntimeError("CUDA out of memory. See logs/error_traceback.txt and reduce memory settings.") from exc\n    except Exception:\n        atomic_write_text(args.output_dir / "logs" / "error_traceback.txt", traceback.format_exc())\n        atomic_json(status_path, status_payload("failed", args, int(latest_metrics.get("step", 0)), latest_metrics, best_metrics, gate_metrics, "Run raised an exception; see logs/error_traceback.txt"))\n        raise\n    final_status = "passed" if gate_metrics else "failed"\n    reason = None if gate_metrics else "No single checkpoint met CER <= 0.05, WER <= 0.05, and empty rate < 0.10"\n    atomic_json(status_path, status_payload(final_status, args, int(latest_metrics.get("step", requested_steps)), latest_metrics, best_metrics, gate_metrics, reason))\n    LOGGER.info("GATE 1 %s", "PASSED" if gate_metrics else "FAILED")\n    return 0 if gate_metrics else 2\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--manifest", type=Path, required=True, help="Exactly 32 manually verified rows")\n    parser.add_argument("--processor-path", type=Path, help="Validated Bengali 73-token MMS processor directory")\n    parser.add_argument("--project-root", type=Path, default=ROOT)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--model-name", default="facebook/mms-300m")\n    parser.add_argument("--trainable-encoder-layers", type=int, default=None)\n    parser.add_argument("--head-lr", type=float, default=1e-3)\n    parser.add_argument("--encoder-lr", type=float, default=1e-5)\n    parser.add_argument("--weight-decay", type=float, default=0.01)\n    parser.add_argument("--warmup-steps", type=int, default=50)\n    parser.add_argument("--batch-size", type=int, default=None)\n    parser.add_argument("--gradient-accumulation-steps", type=int, default=None)\n    parser.add_argument("--max-steps", type=int, default=3000)\n    parser.add_argument("--eval-steps", type=int, default=25)\n    parser.add_argument("--max-audio-seconds", type=float, default=30.0)\n    parser.add_argument("--num-workers", type=int, default=None)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--fp16", dest="precision", action="store_const", const="fp16", default=None)\n    parser.add_argument("--bf16", dest="precision", action="store_const", const="bf16")\n    parser.add_argument("--no-mixed-precision", dest="precision", action="store_const", const="none")\n    parser.add_argument("--gradient-checkpointing", dest="gradient_checkpointing", action="store_true", default=None)\n    parser.add_argument("--no-gradient-checkpointing", dest="gradient_checkpointing", action="store_false")\n    parser.add_argument("--strict-deterministic", action="store_true")\n    parser.add_argument("--manually-verified", action="store_true")\n    parser.add_argument("--allow-cpu", action="store_true")\n    parser.add_argument("--dry-run", action="store_true")\n    parser.add_argument("--smoke-test", action="store_true")\n    return parser\n\n\ndef main() -> int:\n    parser = build_parser()\n    args = parser.parse_args()\n    try:\n        return run_experiment(args)\n    except KeyboardInterrupt:\n        LOGGER.error("Interrupted")\n        return 130\n    except Exception as exc:\n        LOGGER.error("%s", exc)\n        return 1\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/asr_dialect_benchmark/__init__.py': '"""Research package for Bengali dialect-aware ASR."""\n', 'src/asr_dialect_benchmark/modeling/__init__.py': '"""Modeling components for the Bengali dialect ASR project."""\n\nfrom .asr_model import BengaliDialectASR\n\n__all__ = ["BengaliDialectASR"]\n', 'src/asr_dialect_benchmark/modeling/asr_model.py': '"""MMS-300M CTC baseline and dialect-aware MoE model."""\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nfrom transformers import AutoModel\n\nfrom .moe import SparseMixtureOfExperts, masked_mean\n\n\ndef _value(config, name, default=None):\n    if isinstance(config, dict):\n        return config.get(name, default)\n    return getattr(config, name, default)\n\n\nclass BengaliDialectASR(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        model_config = _value(config, "model", config)\n        self.pretrained_model = _value(model_config, "pretrained_model", "facebook/mms-300m")\n        self.num_dialects = int(_value(model_config, "num_dialects", 4))\n        self.use_moe = bool(_value(model_config, "use_moe", True))\n        self.encoder = AutoModel.from_pretrained(self.pretrained_model)\n        if bool(_value(model_config, "gradient_checkpointing", True)):\n            self.encoder.gradient_checkpointing_enable()\n        hidden_size = int(self.encoder.config.hidden_size)\n        if self.use_moe:\n            self.moe = SparseMixtureOfExperts(\n                hidden_size=hidden_size,\n                num_dialects=self.num_dialects,\n                top_k=int(_value(model_config, "top_k", 2)),\n                dropout=float(_value(model_config, "dropout", 0.1)),\n                use_router=bool(_value(model_config, "use_router", True)),\n                use_shared_expert=bool(_value(model_config, "use_shared_expert", True)),\n            )\n        else:\n            self.moe = None\n        self.dialect_classifier = nn.Linear(hidden_size, self.num_dialects)\n        self.ctc_head = nn.Linear(hidden_size, int(_value(model_config, "num_tokens", 64)))\n\n    def feature_lengths(self, input_lengths: torch.Tensor) -> torch.Tensor:\n        return self.encoder._get_feat_extract_output_lengths(input_lengths).to(torch.long)\n\n    def set_phase(self, phase: int, top_layers: int = 4) -> None:\n        for parameter in self.encoder.parameters():\n            parameter.requires_grad = False\n        if phase >= 2:\n            layers = self.encoder.encoder.layers\n            for layer in layers[-top_layers:]:\n                for parameter in layer.parameters():\n                    parameter.requires_grad = True\n            # The final normalization is part of the top representation.\n            if hasattr(self.encoder.encoder, "layer_norm"):\n                for parameter in self.encoder.encoder.layer_norm.parameters():\n                    parameter.requires_grad = True\n\n    def forward(self, input_values=None, attention_mask=None, input_lengths=None, routing_inputs=None):\n        if routing_inputs is not None:\n            if self.moe is None:\n                return {"gate_probs": None, "topk_indices": None}\n            gate_probs, _, topk_indices = self.moe.route(routing_inputs)\n            return {"gate_probs": gate_probs, "topk_indices": topk_indices}\n        encoded = self.encoder(input_values=input_values, attention_mask=attention_mask)\n        hidden_states = encoded.last_hidden_state\n        if input_lengths is None:\n            input_lengths = attention_mask.sum(-1) if attention_mask is not None else input_values.new_full((input_values.shape[0],), input_values.shape[1], dtype=torch.long)\n        output_lengths = self.feature_lengths(input_lengths)\n        output_lengths = output_lengths.clamp(max=hidden_states.shape[1])\n        time = torch.arange(hidden_states.shape[1], device=hidden_states.device)[None, :]\n        feature_mask = time < output_lengths[:, None]\n        if self.moe is not None:\n            hidden_states, gate_probs, topk_indices, router_input = self.moe(hidden_states, feature_mask)\n        else:\n            gate_probs, topk_indices, router_input = None, None, None\n        pooled = masked_mean(hidden_states, feature_mask)\n        return {\n            "logits": self.ctc_head(hidden_states),\n            "dialect_logits": self.dialect_classifier(pooled),\n            "gate_probs": gate_probs,\n            "topk_indices": topk_indices,\n            "router_input": router_input,\n            "output_lengths": output_lengths,\n        }\n', 'src/asr_dialect_benchmark/modeling/experts.py': 'import torch\nimport torch.nn as nn\n\n\nclass ResidualFeedForward(nn.Module):\n    """Lightweight residual feed-forward expert block."""\n\n    def __init__(self, hidden_size: int, ff_dim: int = 512, dropout: float = 0.1):\n        super().__init__()\n        self.fc1 = nn.Linear(hidden_size, ff_dim)\n        self.act = nn.GELU()\n        self.dropout = nn.Dropout(dropout)\n        self.fc2 = nn.Linear(ff_dim, hidden_size)\n        self.norm = nn.LayerNorm(hidden_size)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        residual = x\n        x = self.fc1(x)\n        x = self.act(x)\n        x = self.dropout(x)\n        x = self.fc2(x)\n        x = self.dropout(x)\n        return self.norm(residual + x)\n\n\nclass SharedExpert(nn.Module):\n    def __init__(self, hidden_size: int, ff_dim: int = 512, dropout: float = 0.1):\n        super().__init__()\n        self.block = ResidualFeedForward(hidden_size, ff_dim=ff_dim, dropout=dropout)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.block(x)\n\n\nclass DialectExpert(nn.Module):\n    def __init__(self, hidden_size: int, ff_dim: int = 512, dropout: float = 0.1):\n        super().__init__()\n        self.block = ResidualFeedForward(hidden_size, ff_dim=ff_dim, dropout=dropout)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.block(x)\n', 'src/asr_dialect_benchmark/modeling/moe.py': '"""Utterance-routed sparse dialect mixture of experts."""\n\nimport torch\nimport torch.nn as nn\n\nfrom .experts import DialectExpert, SharedExpert\nfrom .router import DialectRouter\n\n\ndef masked_mean(hidden_states: torch.Tensor, mask: torch.Tensor | None) -> torch.Tensor:\n    if mask is None:\n        return hidden_states.mean(dim=1)\n    weights = mask.to(hidden_states.dtype).unsqueeze(-1)\n    return (hidden_states * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)\n\n\nclass SparseMixtureOfExperts(nn.Module):\n    def __init__(self, hidden_size: int, num_dialects: int = 4, top_k: int = 2, dropout: float = 0.1, use_router: bool = True, use_shared_expert: bool = True):\n        super().__init__()\n        self.use_router = use_router\n        self.use_shared_expert = use_shared_expert\n        self.top_k = min(top_k, num_dialects)\n        self.router = DialectRouter(hidden_size, num_dialects=num_dialects, dropout=dropout) if use_router else None\n        self.shared_expert = SharedExpert(hidden_size=hidden_size, dropout=dropout) if use_shared_expert else None\n        self.dialect_experts = nn.ModuleList(DialectExpert(hidden_size=hidden_size, dropout=dropout) for _ in range(num_dialects))\n\n    def route(self, pooled: torch.Tensor):\n        if self.router is None:\n            gate_probs = pooled.new_full((pooled.shape[0], len(self.dialect_experts)), 1.0 / len(self.dialect_experts))\n        else:\n            gate_probs = self.router(pooled)\n        topk_values, topk_indices = torch.topk(gate_probs, self.top_k, dim=-1)\n        topk_values = topk_values / topk_values.sum(dim=-1, keepdim=True).clamp_min(1e-9)\n        return gate_probs, topk_values, topk_indices\n\n    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor | None = None):\n        pooled = masked_mean(hidden_states, attention_mask)\n        gate_probs, topk_values, topk_indices = self.route(pooled)\n        fusion = torch.zeros_like(pooled)\n        # Dispatch only selected samples to each expert. Top-1 therefore\n        # performs half the dialect-expert work of top-2.\n        for expert_id, expert in enumerate(self.dialect_experts):\n            sample_indices, slots = torch.where(topk_indices == expert_id)\n            if sample_indices.numel() == 0:\n                continue\n            expert_output = expert(pooled.index_select(0, sample_indices))\n            weighted = expert_output * topk_values[sample_indices, slots].unsqueeze(-1)\n            fusion = fusion.index_add(0, sample_indices, weighted)\n        if self.shared_expert is not None:\n            fusion = fusion + self.shared_expert(pooled)\n        return hidden_states + fusion.unsqueeze(1), gate_probs, topk_indices, pooled\n', 'src/asr_dialect_benchmark/modeling/router.py': 'import torch\nimport torch.nn as nn\n\n\nclass DialectRouter(nn.Module):\n    """Predicts dialect probabilities over four dialects."""\n\n    def __init__(self, input_dim: int, num_dialects: int = 4, dropout: float = 0.1):\n        super().__init__()\n        self.input_dim = input_dim\n        self.num_dialects = num_dialects\n        self.proj1 = nn.Linear(input_dim, max(64, input_dim // 2))\n        self.act = nn.GELU()\n        self.dropout = nn.Dropout(dropout)\n        self.proj2 = nn.Linear(max(64, input_dim // 2), num_dialects)\n\n    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:\n        if hidden_states.dim() == 3:\n            x = hidden_states.mean(dim=1)\n        else:\n            x = hidden_states\n        x = self.proj1(x)\n        x = self.act(x)\n        x = self.dropout(x)\n        x = self.proj2(x)\n        return torch.softmax(x, dim=-1)\n'}

for relative_name, source_text in EMBEDDED_SOURCES.items():
    destination = REPO_DIR / relative_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source_text, encoding="utf-8")
print("Embedded diagnostic source snapshot:", REPO_COMMIT)
print("No HF token is required for this notebook; the MMS model is public.")

INPUT_ROOT = Path("/kaggle/input")
DATASET_SLUG = "four-dialect-data-undersampled"
def is_dataset_root(path: Path) -> bool:
    return all((path / split).is_dir() or (path / f"{split}.zip").is_file() for split in ("train", "validation", "test"))

# Prefer the exact Kaggle mount before using a shallow fallback.  This avoids
# recursively walking every WAV file in the attached dataset.
known_data_roots = [
    INPUT_ROOT / "datasets" / "diyalibiswas" / "four-dialect-data-undersampled",
    INPUT_ROOT / "four-dialect-data-undersampled",
    INPUT_ROOT / "four_dialect_data_undersampled",
]
dataset_candidates = [path for path in known_data_roots if path.is_dir() and is_dataset_root(path)]
if not dataset_candidates:
    dataset_candidates = [
        path for path in INPUT_ROOT.iterdir()
        if path.is_dir() and DATASET_SLUG in str(path).lower() and is_dataset_root(path)
    ]
if not dataset_candidates:
    dataset_candidates = [
        path for path in INPUT_ROOT.iterdir()
        if path.is_dir() and is_dataset_root(path)
    ]
if not dataset_candidates:
    raise FileNotFoundError("Attach diyalibiswas/four-dialect-data-undersampled to notebook input")
DATA_ROOT = sorted(dataset_candidates, key=lambda path: len(str(path)))[0]
print("DATA_ROOT=", DATA_ROOT)

CHECKPOINT_OVERRIDE = os.environ.get("CTC_CHECKPOINT", "").strip()
CHECKPOINT_ROOT = Path(CHECKPOINT_OVERRIDE) if CHECKPOINT_OVERRIDE else None
if CHECKPOINT_ROOT and not CHECKPOINT_ROOT.exists():
    raise FileNotFoundError(CHECKPOINT_ROOT)
if CHECKPOINT_ROOT is None:
    checkpoint_candidates = []
    known_output_roots = [
        INPUT_ROOT / "datasets" / "diyalibiswas" / "output",
        INPUT_ROOT / "output",
    ]
    for output_root in [path for path in known_output_roots if path.is_dir()]:
        for config_path in output_root.rglob("config.json"):
            candidate = config_path.parent
            if any((candidate / state_name).is_file() for state_name in ("model.safetensors", "model_state.pt", "pytorch_model.bin")):
                checkpoint_candidates.append(candidate)
    if not checkpoint_candidates:
        raise FileNotFoundError(
            "No checkpoint found below /kaggle/input/datasets/diyalibiswas/output; attach the output dataset"
        )
    def checkpoint_step(path):
        state_path = path / "trainer_state.json"
        if not state_path.is_file():
            return -1
        try:
            return int(json.loads(state_path.read_text(encoding="utf-8")).get("global_step", -1))
        except Exception:
            return -1
    CHECKPOINT_ROOT = sorted(
        checkpoint_candidates,
        key=lambda path: (checkpoint_step(path), str(path)),
    )[-1]
print("CHECKPOINT_ROOT=", CHECKPOINT_ROOT)


In [ ]:
# Build the deterministic 32-row manifest and record the user's explicit
# confirmation that every selected audio/transcript pair was checked.
manifest_path = RUN_DIR / "tiny_manifest.csv"
manifest_command = [
    sys.executable, str(REPO_DIR / "scripts" / "kaggle_ctc_collapse_diagnostics.py"),
    "--data-root", str(DATA_ROOT), "--repo-root", str(REPO_DIR),
    "--output-dir", str(RUN_DIR), "--make-manifest", str(manifest_path),
    "--manifest-count", "32",
]
subprocess.run(manifest_command, check=True)

import csv
with manifest_path.open("r", newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    fieldnames = list(reader.fieldnames or [])
    rows = list(reader)
if len(rows) != 32:
    raise RuntimeError(f"Expected exactly 32 manifest rows, got {len(rows)}")
for row in rows:
    row["manually_verified"] = "YES"
with manifest_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
print(f"Recorded confirmed audio/transcript matches for {len(rows)} rows")
print(manifest_path.read_text(encoding="utf-8")[:1500])


In [ ]:
# Audit the latest attached checkpoint. No training is launched.
audit_log = LOG_DIR / "checkpoint_audit.log"
if CHECKPOINT_ROOT is None:
    status = {
        "status": "blocked",
        "reason": "No attached checkpoint dataset was found",
        "action": "Attach the Kaggle output dataset containing checkpoint files and rerun this cell",
    }
    (RUN_DIR / "checkpoint_audit_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    print(json.dumps(status, indent=2))
else:
    command = [
        sys.executable, str(REPO_DIR / "scripts" / "ctc_collapse_diagnostics.py"),
        "--data-root", str(DATA_ROOT), "--repo-root", str(REPO_DIR),
        "--output-dir", str(RUN_DIR), "--checkpoint", str(CHECKPOINT_ROOT),
        "--sample-count", "100", "--batch-size", "4",
    ]
    with audit_log.open("w", encoding="utf-8") as handle:
        completed = subprocess.run(command, stdout=handle, stderr=subprocess.STDOUT, text=True)
    if completed.returncode:
        print("--- checkpoint_audit.log (tail) ---")
        print(audit_log.read_text(encoding="utf-8", errors="replace")[-20000:])
        raise RuntimeError(f"Checkpoint audit failed; inspect {audit_log}")
    print((RUN_DIR / "ctc_collapse_summary.json").read_text(encoding="utf-8"))


In [ ]:
# Run the one-GPU plain MMS-CTC overfit test. CUDA_VISIBLE_DEVICES=0 makes
# this safe even when Kaggle provisions a T4x2 session.
if CHECKPOINT_ROOT is None:
    raise RuntimeError("No checkpoint was found in the attached output dataset")
tiny_command = [
    sys.executable, str(REPO_DIR / "scripts" / "tiny_overfit_ctc.py"),
    "--manifest", str(manifest_path), "--checkpoint", str(CHECKPOINT_ROOT),
    "--output-dir", str(RUN_DIR / "tiny-overfit"), "--batch-size", "4",
    "--max-steps", "3000", "--eval-every", "50", "--manually-verified",
]
tiny_env = os.environ.copy()
tiny_env["CUDA_VISIBLE_DEVICES"] = "0"
print("Running:", " ".join(map(str, tiny_command)))
completed = subprocess.run(tiny_command, check=False, env=tiny_env)
if completed.returncode:
    raise RuntimeError(f"Tiny overfit failed with exit code {completed.returncode}")
status_path = RUN_DIR / "tiny-overfit" / "tiny_overfit_status.json"
print(status_path.read_text(encoding="utf-8"))


In [ ]:
# Save the confirmed manifest, metrics, logs, and status as one Kaggle output.
manifest = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "repository_commit": REPO_COMMIT,
    "dataset_root": str(DATA_ROOT),
    "checkpoint_root": str(CHECKPOINT_ROOT) if CHECKPOINT_ROOT else None,
    "confirmed_manifest": str(manifest_path),
    "tiny_overfit": True,
    "model_initialized_fresh": True,
    "gpu_visibility": "0",
}
(RUN_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
archive = Path("/kaggle/working/ctc-confirmed-tiny-overfit-outputs.zip")
if archive.exists():
    archive.unlink()
shutil.make_archive(str(archive.with_suffix("")), "zip", RUN_DIR)
print("Saved:", archive)
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(RUN_DIR), path.stat().st_size)
